# Augmentation Baseline Comparison

This notebook reproduces **Table 2** of the paper: a head-to-head comparison of
**CRDA** against five competing augmentation methods, for both the **MLP** and
**XGBoost** base regressors, across the nine benchmark datasets.

For each `(dataset, base model, method, seed)` it:
1. trains the base regressor on the raw training split and records its test MSE;
2. augments the training set with the method under test, retrains, and records the
   augmented test MSE;
3. reports the percentage change `delta_percent = 100 * (aug_mse - mse) / mse`
   (lower = better) and an SDV quality score.

## Methods compared
| Function | Method | Type |
|---|---|---|
| `run_crda`    | **CRDA** (ours)          | residual-preserving counterfactual augmentation |
| `run_tabddpm` | TabDDPM                  | diffusion generative model |
| `run_ctgan`   | CTGAN                    | GAN generative model |
| `run_tvae`    | TVAE                     | VAE generative model |
| `run_c_mixup` | C-Mixup (Yao et al.)     | regression mixup |
| `run_ada`     | ADA (Schneider et al.)   | anchor data augmentation |

## ⚠️ External dependency for C-Mixup and ADA (read before running)
`run_ada` and `run_c_mixup` do **not** run in this repo's environment. They shell out
to the original authors' implementation, which lives in a **separate repo and conda
environment** (the dependencies conflict with ours, so they cannot be merged):

- Upstream code: https://github.com/NoraSchneider/anchordataaugmentation
- It is invoked via `subprocess` inside a conda env named `ada`.

**The path to that repo is currently hard-coded** in the `run_ada` / `run_c_mixup`
cells as:

```python
sys.path.append('/Users/hosseinmohebbi/Desktop/code/anchordataaugmentation')
```

To run C-Mixup/ADA yourself you must (1) clone the upstream repo, (2) create the `ada`
conda env per its README, and (3) **edit that hard-coded path** to point at your local
checkout. If you don't, only those two cells fail — the other four methods still run.

> You do **not** need to re-run C-Mixup/ADA to inspect Table 2: their per-seed results
> are already saved in `comparison.csv` (and aggregated in `aggregated_results.csv`).

## TabDDPM uses cached samples
`run_tabddpm` does **not** train a diffusion model here. It loads pre-generated
synthetic samples from `./diffusion_data/synthetic_data_exp_{seed}/{dataset}.csv`.
Keep the `diffusion_data/` folder to reproduce the TabDDPM column.

## Reproducibility
- A fixed list of **10 seeds** (`SEEDS`) is used; `set_seed()` seeds `random`, `numpy`,
  and `torch` (with `cudnn.deterministic=True`).
- CRDA, C-Mixup, ADA, and the cached TabDDPM column are deterministic given a seed.
- **CTGAN and TVAE are trained live** (`epochs=50`), so their numbers can vary slightly
  across library/hardware versions even with seeding — expect minor drift here.

## How to run / outputs
- Run this notebook **from the `experiments_all_baselines/` directory** (cell 1 adds the
  parent on `sys.path` and dataset paths are relative, e.g. `../data/...`).
- Per-(dataset, model, method, seed) rows are written to **`comparison.csv`**.
- To produce the aggregated means ± standard errors used in the paper, run:
  ```bash
  python ../scripts/collect_baseline_results.py
  ```
  which writes **`aggregated_results.csv`** and **`results_summary.csv`**.


In [1]:
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

from src.utils.config import Config
from src.utils.logger import Logger
from src.experiment import Experiment

import logging
import pandas as pd
import numpy as np
import torch
import random
from src.baseline import BaselineRegressor
from src.dataset import AbstractDataset
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.evaluation.single_table import evaluate_quality
from sdv.evaluation.single_table import get_column_plot
from sdv.metadata import Metadata

comparison_df = pd.DataFrame(columns=['dataset_name', 'method', 'baseline', 'seed', 'mse', 'aug_mse', 'delta_percent', 'quality_score'])
SEEDS = [985772, 305711, 435829, 117952, 963395, 152315, 882371, 359783, 304137, 122579]

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def add_row(dataset_name, method, baseline, seed, mse, aug_mse, delta_percent, quality_score):
    row_data = {
        'dataset_name': dataset_name,
        'method': method,
        'baseline': baseline,
        'seed': seed,
        'mse': mse,
        'aug_mse': aug_mse,
        'delta_percent': delta_percent,
        'quality_score': quality_score
    }
    global comparison_df
    comparison_df = pd.concat([comparison_df, pd.DataFrame([row_data])], ignore_index=True)

In [3]:
def get_dataset_name(config):
    return os.path.basename(config.dataset_path).split('.')[0]

def get_train_data(config):
    if config.sample_sizes is not None:
        assert len(config.sample_sizes) == 1, "sample_sizes should be a list of one element"
        sample_size = config.sample_sizes[0]
    else:
        sample_size = None

    dataset_name = get_dataset_name(config)
    dataset = AbstractDataset(dataset_name, config.dataset_path, seed=config.random_seed, sample_size=sample_size)
    dataset.preprocess()
    X_train, X_test, y_train, y_test = dataset.split(test_size=config.test_size, seed=config.random_seed)

    train_df = pd.DataFrame(X_train, columns=dataset.X.columns)
    train_df['target'] = y_train

    return X_train, X_test, y_train, y_test, train_df

In [4]:
def train_and_evaluate_baseline(config, X_train, y_train, X_test, y_test):
    baseline = BaselineRegressor(config.baseline)
    baseline.set_params(random_state=config.random_seed)
    baseline.train(X_train, y_train)
    mse = baseline.evaluate(X_test, y_test, metric="mse")
    return mse

In [5]:
def run_tvae(config, mse_baseline, train_df, X_test, y_test, metadata):
    set_seed(config.random_seed)

    train_df_len = len(train_df)
    aug_data_len = int(train_df_len * config.aug_data_size_factor)

    # train tvae
    synthesizer = TVAESynthesizer(metadata=metadata, epochs=50)
    synthesizer.fit(train_df)

    # generate aug data
    aug_data = synthesizer.sample(num_rows=aug_data_len)
    aug_train_df = pd.concat([train_df, aug_data], ignore_index=True)
    aug_X_train, aug_y_train = aug_train_df.iloc[:, :-1].to_numpy(), aug_train_df.iloc[:, -1:].values.ravel()

    # train a baseline model on the aug data
    new_baseline = BaselineRegressor(config.baseline)
    new_baseline.train(aug_X_train, aug_y_train)
    aug_mse = new_baseline.evaluate(X_test, y_test, metric="mse")

    delta_percent = ((aug_mse - mse_baseline) / mse_baseline) * 100

    quality_report = evaluate_quality(train_df, aug_train_df, metadata)
    score = quality_report.get_score()
    
    return aug_mse, delta_percent, score

In [6]:
def run_ctgan(config, mse_baseline, train_df, X_test, y_test, metadata):
    set_seed(config.random_seed)

    train_df_len = len(train_df)
    aug_data_len = int(train_df_len * config.aug_data_size_factor)

    # train ctgan
    synthesizer = CTGANSynthesizer(metadata=metadata, epochs=50)
    synthesizer.fit(train_df)

    # generate aug data
    aug_data = synthesizer.sample(num_rows=aug_data_len)
    aug_train_df = pd.concat([train_df, aug_data], ignore_index=True)
    aug_X_train, aug_y_train = aug_train_df.iloc[:, :-1].to_numpy(), aug_train_df.iloc[:, -1:].values.ravel()

    # train a baseline model on the aug data
    new_baseline = BaselineRegressor(config.baseline)
    new_baseline.train(aug_X_train, aug_y_train)
    aug_mse = new_baseline.evaluate(X_test, y_test, metric="mse")

    delta_percent = ((aug_mse - mse_baseline) / mse_baseline) * 100

    quality_report = evaluate_quality(train_df, aug_train_df, metadata)
    score = quality_report.get_score()

    return aug_mse, delta_percent, score

In [7]:
def run_tabddpm(config):
    set_seed(config.random_seed)

    X_train, X_test, y_train, y_test, train_df = get_train_data(config)
    mse_baseline = train_and_evaluate_baseline(config, X_train, y_train, X_test, y_test)
    dataset_name = get_dataset_name(config)
    metadata = Metadata.detect_from_dataframe(data=train_df,table_name=get_dataset_name(config))

    # load synthetic data from diffusion_data folder its a csv and we need the panda
    synthetic_data = pd.read_csv(f"./diffusion_data/synthetic_data_exp_{config.random_seed}/{dataset_name}.csv")
    # replace the headers in the synthetic data with the headers in the train_df
    # synthetic_data.columns = train_df.columns
    aug_train_df = pd.concat([train_df, synthetic_data], ignore_index=True)
    aug_X_train, aug_y_train = aug_train_df.iloc[:, :-1].to_numpy(), aug_train_df.iloc[:, -1:].values.ravel()

    new_baseline = BaselineRegressor(config.baseline)
    new_baseline.train(aug_X_train, aug_y_train)
    aug_mse = new_baseline.evaluate(X_test, y_test, metric="mse")
    delta_percent = ((aug_mse - mse_baseline) / mse_baseline) * 100
    quality_report = evaluate_quality(train_df, aug_train_df, metadata)
    score = quality_report.get_score()

    return mse_baseline, aug_mse, delta_percent, score

In [8]:
def run_ada(config, mse_baseline, train_df, X_train, y_train, X_test, y_test, metadata):
    """Run ADA augmentation by calling into the ada conda environment."""
    import subprocess
    import pickle
    import tempfile
    import os
    
    set_seed(config.random_seed)
    
    train_len = len(y_train)
    aug_data_len = int(train_len * config.aug_data_size_factor)
    
    # Create temporary files for data transfer
    with tempfile.NamedTemporaryFile(mode='wb', suffix='.pkl', delete=False) as f_input:
        input_file = f_input.name
        pickle.dump({'X': X_train, 'y': y_train, 'seed': config.random_seed, 
                     'aug_data_len': aug_data_len}, f_input)
    
    with tempfile.NamedTemporaryFile(mode='wb', suffix='.pkl', delete=False) as f_output:
        output_file = f_output.name
    
    # Create a temporary Python script to run in ada environment
    script_content = f"""
import sys
sys.path.append('/Users/hosseinmohebbi/Desktop/code/anchordataaugmentation')
import pickle
import numpy as np
from ada_augmented_data import get_ada_augmented_data

# Load input data
with open('{input_file}', 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
seed = data['seed']
aug_data_len = data['aug_data_len']

# Get augmented data
X_ada, y_ada = get_ada_augmented_data(X, y, gammas=[0.5, 0.75, 1.5, 2.0], num_anchors=10, seed=seed)

# Sample to match desired augmentation size
if len(X_ada) > aug_data_len:
    indices = np.random.choice(len(X_ada), size=aug_data_len, replace=False)
    X_ada = X_ada[indices]
    y_ada = y_ada[indices]

# Save output
with open('{output_file}', 'wb') as f:
    pickle.dump({{'X_ada': X_ada, 'y_ada': y_ada}}, f)
"""
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f_script:
        script_file = f_script.name
        f_script.write(script_content)
    
    try:
        # Run the script in the ada conda environment
        cmd = f"bash -c 'source $(conda info --base)/etc/profile.d/conda.sh && conda activate ada && python {script_file}'"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable='/bin/bash')
        
        if result.returncode != 0:
            print(f"Error running ADA: {result.stderr}")
            raise RuntimeError(f"ADA augmentation failed: {result.stderr}")
        
        # Load the augmented data
        with open(output_file, 'rb') as f:
            output_data = pickle.load(f)
        
        X_ada = output_data['X_ada']
        y_ada = output_data['y_ada']
        
        # Create augmented dataframe
        aug_train_df = pd.DataFrame(X_ada, columns=train_df.columns[:-1])
        aug_train_df['target'] = y_ada

        # Combine with original training data
        combined_df = pd.concat([train_df, aug_train_df], ignore_index=True)
        aug_X_train, aug_y_train = combined_df.iloc[:, :-1].to_numpy(), combined_df.iloc[:, -1:].values.ravel()

        # Train baseline on augmented data
        new_baseline = BaselineRegressor(config.baseline)
        new_baseline.train(aug_X_train, aug_y_train)
        aug_mse = new_baseline.evaluate(X_test, y_test, metric="mse")
        
        delta_percent = ((aug_mse - mse_baseline) / mse_baseline) * 100
        
        quality_report = evaluate_quality(train_df, combined_df, metadata)
        score = quality_report.get_score()
        
        return aug_mse, delta_percent, score
        
    finally:
        # Clean up temporary files
        for temp_file in [input_file, output_file, script_file]:
            if os.path.exists(temp_file):
                os.remove(temp_file)


In [9]:
def run_c_mixup(config, mse_baseline, train_df, X_train, y_train, X_test, y_test, metadata):
    """Run C-Mixup augmentation by calling into the ada conda environment."""
    import subprocess
    import pickle
    import tempfile
    import os
    
    set_seed(config.random_seed)
    
    train_len = len(y_train)
    aug_data_len = int(train_len * config.aug_data_size_factor)
    k = int(np.ceil(config.aug_data_size_factor))
    
    # Create temporary files for data transfer
    with tempfile.NamedTemporaryFile(mode='wb', suffix='.pkl', delete=False) as f_input:
        input_file = f_input.name
        pickle.dump({'X': X_train, 'y': y_train, 'seed': config.random_seed, 'k': k,
                     'aug_data_len': aug_data_len}, f_input)
    
    with tempfile.NamedTemporaryFile(mode='wb', suffix='.pkl', delete=False) as f_output:
        output_file = f_output.name
    
    # Create a temporary Python script to run in ada environment
    script_content = f"""
import sys
sys.path.append('/Users/hosseinmohebbi/Desktop/code/anchordataaugmentation')
import pickle
import numpy as np
from c_mixup_augmented_data import get_cmixup_augmented_data

# Load input data
with open('{input_file}', 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
seed = data['seed']
k = data['k']
aug_data_len = data['aug_data_len']

# Get augmented data
X_c_mixup, y_c_mixup = get_cmixup_augmented_data(X, y, k=k, seed=seed)

# Sample to match desired augmentation size
if len(X_c_mixup) > aug_data_len:
    indices = np.random.choice(len(X_c_mixup), size=aug_data_len, replace=False)
    X_c_mixup = X_c_mixup[indices]
    y_c_mixup = y_c_mixup[indices]

# Save output
with open('{output_file}', 'wb') as f:
    pickle.dump({{'X_c_mixup': X_c_mixup, 'y_c_mixup': y_c_mixup}}, f)
"""
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f_script:
        script_file = f_script.name
        f_script.write(script_content)
    
    try:
        # Run the script in the ada conda environment
        cmd = f"bash -c 'source $(conda info --base)/etc/profile.d/conda.sh && conda activate ada && python {script_file}'"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable='/bin/bash')
        
        if result.returncode != 0:
            print(f"Error running C-Mixup: {result.stderr}")
            raise RuntimeError(f"C-Mixup augmentation failed: {result.stderr}")
        
        # Load the augmented data
        with open(output_file, 'rb') as f:
            output_data = pickle.load(f)
        
        X_c_mixup = output_data['X_c_mixup']
        y_c_mixup = output_data['y_c_mixup']
        
        # Create augmented dataframe
        aug_train_df = pd.DataFrame(X_c_mixup, columns=train_df.columns[:-1])
        aug_train_df['target'] = y_c_mixup

        # Combine with original training data
        combined_df = pd.concat([train_df, aug_train_df], ignore_index=True)
        aug_X_train, aug_y_train = combined_df.iloc[:, :-1].to_numpy(), combined_df.iloc[:, -1:].values.ravel()
        
        # Train baseline on augmented data
        new_baseline = BaselineRegressor(config.baseline)
        new_baseline.train(aug_X_train, aug_y_train)
        aug_mse = new_baseline.evaluate(X_test, y_test, metric="mse")
        
        delta_percent = ((aug_mse - mse_baseline) / mse_baseline) * 100
        
        quality_report = evaluate_quality(train_df, combined_df, metadata)
        score = quality_report.get_score()
        
        return aug_mse, delta_percent, score
        
    finally:
        # Clean up temporary files
        for temp_file in [input_file, output_file, script_file]:
            if os.path.exists(temp_file):
                os.remove(temp_file)


In [10]:
def run_crda(config, train_df, metadata):
    logger = Logger(log_to_file=False, log_to_console=False, log_level=logging.WARNING)
    experiment = Experiment(config, logger)
    results_df = experiment.run()

    assert len(experiment.datasets) == 1
    assert experiment.combined_X_train is not None
    assert experiment.combined_y_train is not None
    
    aug_X_train = experiment.combined_X_train
    aug_y_train = experiment.combined_y_train

    aug_train_df = pd.DataFrame(aug_X_train, columns=experiment.datasets[0].X.columns)
    aug_train_df['target'] = aug_y_train

    mse_row = results_df[results_df['metric'] == 'mse']
    mse = mse_row['mean'].values[0]

    aug_mse_row = results_df[results_df['metric'] == 'aug_mse']
    aug_mse = aug_mse_row['mean'].values[0]

    delta_row = results_df[results_df['metric'] == 'delta_mse']
    delta_percent = delta_row['mean'].values[0]

    quality_report = evaluate_quality(train_df, aug_train_df, metadata)
    score = quality_report.get_score()
    
    return mse, aug_mse, delta_percent, score

In [11]:
def run_all(config):
    X_train, X_test, y_train, y_test, train_df = get_train_data(config)
    mse_baseline = train_and_evaluate_baseline(config, X_train, y_train, X_test, y_test)
    metadata = Metadata.detect_from_dataframe(data=train_df,table_name=get_dataset_name(config))

    print("##################################### Running C-Mixup #####################################")
    aug_mse_c_mixup, delta_percent_c_mixup, score_c_mixup = run_c_mixup(config, mse_baseline, train_df, X_train, y_train, X_test, y_test, metadata)
    print("##################################### Running ADA #####################################")
    aug_mse_ada, delta_percent_ada, score_ada = run_ada(config, mse_baseline, train_df, X_train, y_train, X_test, y_test, metadata)
    print("##################################### Running TabDDPM #####################################")
    mse_tabddpm, aug_mse_tabddpm, delta_percent_tabddpm, score_tabddpm = run_tabddpm(config)
    print("##################################### Running CTGAN #####################################")
    aug_mse_ctgan, delta_percent_ctgan, score_ctgan = run_ctgan(config, mse_baseline, train_df, X_test, y_test, metadata)
    print("##################################### Running TVAE #####################################")
    aug_mse_tvae, delta_percent_tvae, score_tvae = run_tvae(config, mse_baseline, train_df, X_test, y_test, metadata)
    print("##################################### Running CRDA #####################################")
    mse_crda, aug_mse_crda, delta_percent_crda, score_crda = run_crda(config, train_df, metadata)
    assert mse_crda == mse_baseline, "mse_crda should be equal to mse_baseline"
    assert mse_tabddpm == mse_baseline, "mse_tabddpm should be equal to mse_baseline"

    print("##################################### Done #####################################")

    add_row(get_dataset_name(config), "c_mixup", config.baseline, config.random_seed, mse_baseline, aug_mse_c_mixup, delta_percent_c_mixup, score_c_mixup)
    add_row(get_dataset_name(config), "ada", config.baseline, config.random_seed, mse_baseline, aug_mse_ada, delta_percent_ada, score_ada)
    add_row(get_dataset_name(config), "tabddpm", config.baseline, config.random_seed, mse_baseline, aug_mse_tabddpm, delta_percent_tabddpm, score_tabddpm)
    add_row(get_dataset_name(config), "tvae", config.baseline, config.random_seed, mse_baseline, aug_mse_tvae, delta_percent_tvae, score_tvae)
    add_row(get_dataset_name(config), "ctgan", config.baseline, config.random_seed, mse_baseline, aug_mse_ctgan, delta_percent_ctgan, score_ctgan)
    add_row(get_dataset_name(config), "crda", config.baseline, config.random_seed, mse_crda, aug_mse_crda, delta_percent_crda, score_crda)

    print(f"mse_c_mixup: {mse_baseline}, aug_mse_c_mixup: {aug_mse_c_mixup}, delta_percent_c_mixup: {delta_percent_c_mixup}, score_c_mixup: {score_c_mixup}")
    print(f"mse_ada: {mse_baseline}, aug_mse_ada: {aug_mse_ada}, delta_percent_ada: {delta_percent_ada}, score_ada: {score_ada}")
    print(f"mse_tabddpm: {mse_baseline}, aug_mse_tabddpm: {aug_mse_tabddpm}, delta_percent_tabddpm: {delta_percent_tabddpm}, score_tabddpm: {score_tabddpm}")
    print(f"mse_ctgan: {mse_baseline}, aug_mse_ctgan: {aug_mse_ctgan}, delta_percent_ctgan: {delta_percent_ctgan}, score_ctgan: {score_ctgan}")
    print(f"mse_tvae: {mse_baseline}, aug_mse_tvae: {aug_mse_tvae}, delta_percent_tvae: {delta_percent_tvae}, score_tvae: {score_tvae}")
    print(f"mse_crda: {mse_crda}, aug_mse_crda: {aug_mse_crda}, delta_percent_crda: {delta_percent_crda}, score_crda: {score_crda}")

In [12]:
def run_comparison(config):
    for seed in SEEDS:
        config.random_seed = seed
        run_all(config)

In [13]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/HousePrice.csv",
    results_dir="../experiments_all_baselines/HousePrice",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 695.23it/s]|
Column Shapes Score: 92.43%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 806.59it/s]|
Column Pair Trends Score: 99.74%

Overall Score (Average): 96.08%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1205.74it/s]|
Column Shapes Score: 95.61%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 869.54it/s]|
Column Pair Trends Score: 99.16%

Overall Score (Average): 97.38%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 888.20it/s]|
Column Shapes Score: 92.15%

(2/2) Evaluating Column Pair Trends: |█████

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 653.22it/s]|
Column Shapes Score: 88.21%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 667.42it/s]|
Column Pair Trends Score: 96.95%

Overall Score (Average): 92.58%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 353.54it/s]|
Column Shapes Score: 76.31%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 843.25it/s]|
Column Pair Trends Score: 92.18%

Overall Score (Average): 84.25%

##################################### Running CRDA #####################################


Best trial: 11. Best value: 0.000409467: 100%|██████████| 30/30 [00:18<00:00,  1.66it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1235.89it/s]|
Column Shapes Score: 96.06%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 628.39it/s]|
Column Pair Trends Score: 99.69%

Overall Score (Average): 97.88%

##################################### Done #####################################
mse_c_mixup: 0.0007095544948962084, aug_mse_c_mixup: 0.00047682791037247205, delta_percent_c_mixup: -32.79897262264245, score_c_mixup: 0.9608461009348246
mse_ada: 0.0007095544948962084, aug_mse_ada: 0.0003499960277349441, delta_percent_ada: -50.673834039182495, score_ada: 0.9738293331276202
mse_tabddpm: 0.0007095544948962084, aug_mse_tabddpm: 0.0003687344778518365, delta_percent_tabddpm: -48.032958637549896, score_tabddpm: 0.9544816933269744
mse_ctgan: 0.0007095544948962084, aug_mse_ctgan: 0.0038621637091749515, delta_percent_ctgan: 444.30825778080623, score_ctgan: 0.9258214988437202
mse_tvae: 0.0007095544948962084, aug_mse_tvae: 0.

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1810.72it/s]|
Column Shapes Score: 98.22%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 779.96it/s]|
Column Pair Trends Score: 99.58%

Overall Score (Average): 98.9%

##################################### Done #####################################
mse_c_mixup: 0.0006139978614678536, aug_mse_c_mixup: 0.0004388915278165234, delta_percent_c_mixup: -28.519046179202046, score_c_mixup: 0.9612883618706087
mse_ada: 0.0006139978614678536, aug_mse_ada: 0.0003517174053372242, delta_percent_ada: -42.71683544688719, score_ada: 0.9758239738429584
mse_tabddpm: 0.0006139978614678536, aug_mse_tabddpm: 0.00046461123198557775, delta_percent_tabddpm: -24.330154688999205, score_tabddpm: 0.9576847205963352
mse_ctgan: 0.0006139978614678536, aug_mse_ctgan: 0.004276951037246803, delta_percent_ctgan: 596.5742563047554, score_ctgan: 0.9270522759888955
mse_tvae: 0.0006139978614678536, aug_mse_tvae: 0.000

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 583.65it/s]|
Column Shapes Score: 86.9%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 729.39it/s]|
Column Pair Trends Score: 96.85%

Overall Score (Average): 91.87%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 346.49it/s]|
Column Shapes Score: 75.29%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 665.88it/s]|
Column Pair Trends Score: 91.22%

Overall Score (Average): 83.25%

##################################### Running CRDA #####################################


Best trial: 27. Best value: 0.000448412: 100%|██████████| 30/30 [00:15<00:00,  1.92it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1601.57it/s]|
Column Shapes Score: 98.26%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 624.72it/s]|
Column Pair Trends Score: 99.68%

Overall Score (Average): 98.97%

##################################### Done #####################################
mse_c_mixup: 0.00031410242191919524, aug_mse_c_mixup: 0.0005319195993774186, delta_percent_c_mixup: 69.34590829556169, score_c_mixup: 0.9658692354594554
mse_ada: 0.00031410242191919524, aug_mse_ada: 0.00032406283365659787, delta_percent_ada: 3.171071294689031, score_ada: 0.9737386101693886
mse_tabddpm: 0.00031410242191919524, aug_mse_tabddpm: 0.0003470344894688937, delta_percent_tabddpm: 10.484499720976508, score_tabddpm: 0.9541274317795663
mse_ctgan: 0.00031410242191919524, aug_mse_ctgan: 0.006171348988721254, delta_percent_ctgan: 1864.756893949984, score_ctgan: 0.9187218694610307
mse_tvae: 0.00031410242191919524, aug_mse_tvae: 0.0

In [14]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/623_fri_c4_1000_10.csv",
    results_dir="../experiments_all_baselines/623_fri_c4_1000_10",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1026.51it/s]|
Column Shapes Score: 93.91%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 821.54it/s]|
Column Pair Trends Score: 99.71%

Overall Score (Average): 96.81%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1713.17it/s]|
Column Shapes Score: 98.32%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 865.06it/s]|
Column Pair Trends Score: 99.23%

Overall Score (Average): 98.78%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1668.68it/s]|
Column Shapes Score: 97.77%

(2/2) Evaluating Column Pair Trends

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 594.15it/s]|
Column Shapes Score: 88.86%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 739.30it/s]|
Column Pair Trends Score: 94.91%

Overall Score (Average): 91.88%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 418.71it/s]|
Column Shapes Score: 79.8%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 873.50it/s]|
Column Pair Trends Score: 93.58%

Overall Score (Average): 86.69%

##################################### Running CRDA #####################################


Best trial: 7. Best value: 0.00110452: 100%|██████████| 30/30 [00:22<00:00,  1.36it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1568.55it/s]|
Column Shapes Score: 98.21%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 793.81it/s]|
Column Pair Trends Score: 99.59%

Overall Score (Average): 98.9%

##################################### Done #####################################
mse_c_mixup: 0.0011888439141245375, aug_mse_c_mixup: 0.005165038548118508, delta_percent_c_mixup: 334.45892995314136, score_c_mixup: 0.9681163199378804
mse_ada: 0.0011888439141245375, aug_mse_ada: 0.0015734801384247998, delta_percent_ada: 32.35380353387329, score_ada: 0.9877836763287406
mse_tabddpm: 0.0011888439141245375, aug_mse_tabddpm: 0.002056461039708579, delta_percent_tabddpm: 72.97990217857601, score_tabddpm: 0.9828355709602781
mse_ctgan: 0.0011888439141245375, aug_mse_ctgan: 0.004817277546424172, delta_percent_ctgan: 305.2068979948143, score_ctgan: 0.9188313296074868
mse_tvae: 0.0011888439141245375, aug_mse_tvae: 0.00266137

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1729.55it/s]|
Column Shapes Score: 98.3%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 855.21it/s]|
Column Pair Trends Score: 99.38%

Overall Score (Average): 98.84%

##################################### Done #####################################
mse_c_mixup: 0.0015062705960588463, aug_mse_c_mixup: 0.0034264649457533525, delta_percent_c_mixup: 127.48003942443613, score_c_mixup: 0.975549092771895
mse_ada: 0.0015062705960588463, aug_mse_ada: 0.0017817202317865948, delta_percent_ada: 18.28686269575081, score_ada: 0.988141625226594
mse_tabddpm: 0.0015062705960588463, aug_mse_tabddpm: 0.001980225704291178, delta_percent_tabddpm: 31.46546905134005, score_tabddpm: 0.9835565747849175
mse_ctgan: 0.0015062705960588463, aug_mse_ctgan: 0.004348587463268224, delta_percent_ctgan: 188.6989545335542, score_ctgan: 0.9214715170664041
mse_tvae: 0.0015062705960588463, aug_mse_tvae: 0.002424814

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 504.96it/s]|
Column Shapes Score: 87.59%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 411.85it/s]|
Column Pair Trends Score: 93.87%

Overall Score (Average): 90.73%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 390.65it/s]|
Column Shapes Score: 78.95%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 525.44it/s]|
Column Pair Trends Score: 93.43%

Overall Score (Average): 86.19%

##################################### Running CRDA #####################################


Best trial: 22. Best value: 0.00124961: 100%|██████████| 30/30 [00:23<00:00,  1.28it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1539.14it/s]|
Column Shapes Score: 98.92%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 765.67it/s]|
Column Pair Trends Score: 99.76%

Overall Score (Average): 99.34%

##################################### Done #####################################
mse_c_mixup: 0.0014868818228540998, aug_mse_c_mixup: 0.002319523553753761, delta_percent_c_mixup: 55.99918689579436, score_c_mixup: 0.9808516961969251
mse_ada: 0.0014868818228540998, aug_mse_ada: 0.0020235008210277806, delta_percent_ada: 36.090225189761874, score_ada: 0.9863672034398263
mse_tabddpm: 0.0014868818228540998, aug_mse_tabddpm: 0.002021792569543609, delta_percent_tabddpm: 35.97533700847437, score_tabddpm: 0.9854782182194162
mse_ctgan: 0.0014868818228540998, aug_mse_ctgan: 0.002625450765984777, delta_percent_ctgan: 76.57427279224997, score_ctgan: 0.9072884327635965
mse_tvae: 0.0014868818228540998, aug_mse_tvae: 0.0023473

In [15]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/ConcreteCompressiveStrength.csv",
    results_dir="../experiments_all_baselines/ConcreteCompressiveStrength",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 856.74it/s]|
Column Shapes Score: 92.01%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 828.46it/s]|
Column Pair Trends Score: 99.71%

Overall Score (Average): 95.86%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 972.03it/s]|
Column Shapes Score: 94.15%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 836.81it/s]|
Column Pair Trends Score: 99.44%

Overall Score (Average): 96.79%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 617.37it/s]|
Column Shapes Score: 87.85%

(2/2) Evaluating Column Pair Trends: |██████

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 477.31it/s]|
Column Shapes Score: 88.49%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 805.48it/s]|
Column Pair Trends Score: 94.3%

Overall Score (Average): 91.39%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 385.85it/s]|
Column Shapes Score: 78.49%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 865.86it/s]|
Column Pair Trends Score: 93.25%

Overall Score (Average): 85.87%

##################################### Running CRDA #####################################


Best trial: 16. Best value: 0.00177773: 100%|██████████| 30/30 [00:20<00:00,  1.46it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1157.40it/s]|
Column Shapes Score: 95.75%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 784.26it/s]|
Column Pair Trends Score: 98.91%

Overall Score (Average): 97.33%

##################################### Done #####################################
mse_c_mixup: 0.00313411152229526, aug_mse_c_mixup: 0.003063133763962863, delta_percent_c_mixup: -2.2646851532716594, score_c_mixup: 0.9585734610400833
mse_ada: 0.00313411152229526, aug_mse_ada: 0.0029688160033501747, delta_percent_ada: -5.27407904183421, score_ada: 0.967929247802235
mse_tabddpm: 0.00313411152229526, aug_mse_tabddpm: 0.002797519250628459, delta_percent_tabddpm: -10.739639265302802, score_tabddpm: 0.9343196301958034
mse_ctgan: 0.00313411152229526, aug_mse_ctgan: 0.003497726809293465, delta_percent_ctgan: 11.601861784800564, score_ctgan: 0.9139342331470455
mse_tvae: 0.00313411152229526, aug_mse_tvae: 0.0034614044005238

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1008.54it/s]|
Column Shapes Score: 96.18%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 741.51it/s]|
Column Pair Trends Score: 99.52%

Overall Score (Average): 97.85%

##################################### Done #####################################
mse_c_mixup: 0.002114647643600233, aug_mse_c_mixup: 0.0021911426828833016, delta_percent_c_mixup: 3.6173893799552492, score_c_mixup: 0.9583664774246368
mse_ada: 0.002114647643600233, aug_mse_ada: 0.002020228631043087, delta_percent_ada: -4.464999776340779, score_ada: 0.9711348858353632
mse_tabddpm: 0.002114647643600233, aug_mse_tabddpm: 0.002101103968986979, delta_percent_tabddpm: -0.640469567317385, score_tabddpm: 0.9367214412446252
mse_ctgan: 0.002114647643600233, aug_mse_ctgan: 0.0033902595033749685, delta_percent_ctgan: 60.32266716562666, score_ctgan: 0.8951310236166317
mse_tvae: 0.002114647643600233, aug_mse_tvae: 0.00226336748

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 483.25it/s]|
Column Shapes Score: 84.58%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 818.89it/s]|
Column Pair Trends Score: 94.18%

Overall Score (Average): 89.38%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 407.83it/s]|
Column Shapes Score: 80.56%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 853.41it/s]|
Column Pair Trends Score: 95.1%

Overall Score (Average): 87.83%

##################################### Running CRDA #####################################


Best trial: 14. Best value: 0.00183597: 100%|██████████| 30/30 [00:20<00:00,  1.43it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1443.93it/s]|
Column Shapes Score: 97.8%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 784.74it/s]|
Column Pair Trends Score: 99.5%

Overall Score (Average): 98.65%

##################################### Done #####################################
mse_c_mixup: 0.0022953528394062863, aug_mse_c_mixup: 0.002212325382491739, delta_percent_c_mixup: -3.6171979962794336, score_c_mixup: 0.9640390116087175
mse_ada: 0.0022953528394062863, aug_mse_ada: 0.002237149137488089, delta_percent_ada: -2.5357191678318287, score_ada: 0.9646830130436554
mse_tabddpm: 0.0022953528394062863, aug_mse_tabddpm: 0.002600672089806114, delta_percent_tabddpm: 13.301626014011914, score_tabddpm: 0.9355520956389636
mse_ctgan: 0.0022953528394062863, aug_mse_ctgan: 0.002876498053948425, delta_percent_ctgan: 25.31833906165194, score_ctgan: 0.8937718194725595
mse_tvae: 0.0022953528394062863, aug_mse_tvae: 0.00240270

In [16]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/EnergyEfficiency.csv",
    results_dir="../experiments_all_baselines/EnergyEfficiency",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1140.66it/s]|
Column Shapes Score: 90.86%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 859.68it/s]|
Column Pair Trends Score: 99.84%

Overall Score (Average): 95.35%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1534.97it/s]|
Column Shapes Score: 95.0%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 352.45it/s]|
Column Pair Trends Score: 99.5%

Overall Score (Average): 97.25%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1060.56it/s]|
Column Shapes Score: 90.67%

(2/2) Evaluating Column Pair Trends: 

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 985.71it/s]|
Column Shapes Score: 89.85%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 859.09it/s]|
Column Pair Trends Score: 91.5%

Overall Score (Average): 90.68%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 837.30it/s]|
Column Shapes Score: 86.28%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 884.59it/s]|
Column Pair Trends Score: 94.84%

Overall Score (Average): 90.56%

##################################### Running CRDA #####################################


Best trial: 12. Best value: 0.000487262: 100%|██████████| 30/30 [00:20<00:00,  1.48it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1085.20it/s]|
Column Shapes Score: 94.25%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 764.63it/s]|
Column Pair Trends Score: 99.24%

Overall Score (Average): 96.75%

##################################### Done #####################################
mse_c_mixup: 0.001300772579836663, aug_mse_c_mixup: 0.00060693160265152, delta_percent_c_mixup: -53.340682909557344, score_c_mixup: 0.9534935079616078
mse_ada: 0.001300772579836663, aug_mse_ada: 0.0010753593017723043, delta_percent_ada: -17.329184329259437, score_ada: 0.9725072254448313
mse_tabddpm: 0.001300772579836663, aug_mse_tabddpm: 0.0011422237977846143, delta_percent_tabddpm: -12.18881643953146, score_tabddpm: 0.9503396726367552
mse_ctgan: 0.001300772579836663, aug_mse_ctgan: 0.0007064358457819643, delta_percent_ctgan: -45.69105647424772, score_ctgan: 0.9067531306402176
mse_tvae: 0.001300772579836663, aug_mse_tvae: 0.000826

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1811.25it/s]|
Column Shapes Score: 97.57%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 864.16it/s]|
Column Pair Trends Score: 99.65%

Overall Score (Average): 98.61%

##################################### Done #####################################
mse_c_mixup: 0.0011526985885392895, aug_mse_c_mixup: 0.0007104811848759457, delta_percent_c_mixup: -38.36366315185012, score_c_mixup: 0.9550315108122129
mse_ada: 0.0011526985885392895, aug_mse_ada: 0.0007982101908457225, delta_percent_ada: -30.752913312991737, score_ada: 0.974147113360465
mse_tabddpm: 0.0011526985885392895, aug_mse_tabddpm: 0.001002403484887224, delta_percent_tabddpm: -13.03854321904922, score_tabddpm: 0.9443573217316454
mse_ctgan: 0.0011526985885392895, aug_mse_ctgan: 0.0008151865103454151, delta_percent_ctgan: -29.280167560678017, score_ctgan: 0.8828790291104766
mse_tvae: 0.0011526985885392895, aug_mse_tvae: 0.0

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 894.52it/s]|
Column Shapes Score: 87.21%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 850.18it/s]|
Column Pair Trends Score: 92.05%

Overall Score (Average): 89.63%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 734.75it/s]|
Column Shapes Score: 83.01%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 812.07it/s]|
Column Pair Trends Score: 93.98%

Overall Score (Average): 88.5%

##################################### Running CRDA #####################################


Best trial: 23. Best value: 0.000425877: 100%|██████████| 30/30 [00:19<00:00,  1.54it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 2027.90it/s]|
Column Shapes Score: 99.13%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 858.25it/s]|
Column Pair Trends Score: 99.74%

Overall Score (Average): 99.44%

##################################### Done #####################################
mse_c_mixup: 0.0009737250715466163, aug_mse_c_mixup: 0.0005427926746420108, delta_percent_c_mixup: -44.25606462203279, score_c_mixup: 0.9597379013211836
mse_ada: 0.0009737250715466163, aug_mse_ada: 0.000680067178615679, delta_percent_ada: -30.158193674165723, score_ada: 0.9714104945082972
mse_tabddpm: 0.0009737250715466163, aug_mse_tabddpm: 0.0008821464117682486, delta_percent_tabddpm: -9.404981185593666, score_tabddpm: 0.9524612543688484
mse_ctgan: 0.0009737250715466163, aug_mse_ctgan: 0.0006443014366824987, delta_percent_ctgan: -33.83127789252438, score_ctgan: 0.8962930053056223
mse_tvae: 0.0009737250715466163, aug_mse_tvae: 0.0

In [17]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/WineQuality.csv",
    results_dir="../experiments_all_baselines/WineQuality",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 52.34it/s]|
Column Shapes Score: 94.18%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 726.13it/s]|
Column Pair Trends Score: 99.9%

Overall Score (Average): 97.04%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 124.14it/s]|
Column Shapes Score: 97.59%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 589.42it/s]|
Column Pair Trends Score: 99.67%

Overall Score (Average): 98.63%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 90.16it/s]|
Column Shapes Score: 96.64%

(2/2) Evaluating Column Pair Trends: |███

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 27.21it/s]|
Column Shapes Score: 89.56%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 412.05it/s]|
Column Pair Trends Score: 93.88%

Overall Score (Average): 91.72%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 30.17it/s]|
Column Shapes Score: 89.76%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 653.62it/s]|
Column Pair Trends Score: 97.88%

Overall Score (Average): 93.82%

##################################### Running CRDA #####################################


Best trial: 23. Best value: 0.0118172: 100%|██████████| 30/30 [00:23<00:00,  1.26it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 138.57it/s]|
Column Shapes Score: 98.03%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 731.55it/s]|
Column Pair Trends Score: 99.76%

Overall Score (Average): 98.9%

##################################### Done #####################################
mse_c_mixup: 0.015078899894284384, aug_mse_c_mixup: 0.014884380503614436, delta_percent_c_mixup: -1.2900104917049051, score_c_mixup: 0.9703916627646135
mse_ada: 0.015078899894284384, aug_mse_ada: 0.015090812968787991, delta_percent_ada: 0.07900493130883225, score_ada: 0.9862893182770416
mse_tabddpm: 0.015078899894284384, aug_mse_tabddpm: 0.014552862375363084, delta_percent_tabddpm: -3.488566955210659, score_tabddpm: 0.9607003545486499
mse_ctgan: 0.015078899894284384, aug_mse_ctgan: 0.014384528359486686, delta_percent_ctgan: -4.604921709579743, score_ctgan: 0.9172227042721945
mse_tvae: 0.015078899894284384, aug_mse_tvae: 0.0147999349

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 202.60it/s]|
Column Shapes Score: 98.3%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 747.47it/s]|
Column Pair Trends Score: 99.56%

Overall Score (Average): 98.93%

##################################### Done #####################################
mse_c_mixup: 0.013133458413376504, aug_mse_c_mixup: 0.012691815850626332, delta_percent_c_mixup: -3.362728603917125, score_c_mixup: 0.9749381733579676
mse_ada: 0.013133458413376504, aug_mse_ada: 0.013399205732669431, delta_percent_ada: 2.0234374749476665, score_ada: 0.9862349233911314
mse_tabddpm: 0.013133458413376504, aug_mse_tabddpm: 0.01287909263031779, delta_percent_tabddpm: -1.9367768568836485, score_tabddpm: 0.9595373372644138
mse_ctgan: 0.013133458413376504, aug_mse_ctgan: 0.013507992295122782, delta_percent_ctgan: 2.8517536657733147, score_ctgan: 0.8982832602407893
mse_tvae: 0.013133458413376504, aug_mse_tvae: 0.013223972679

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 23.00it/s]|
Column Shapes Score: 88.88%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 700.04it/s]|
Column Pair Trends Score: 94.07%

Overall Score (Average): 91.48%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 30.16it/s]|
Column Shapes Score: 90.3%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 701.11it/s]|
Column Pair Trends Score: 98.04%

Overall Score (Average): 94.17%

##################################### Running CRDA #####################################


Best trial: 19. Best value: 0.0116272: 100%|██████████| 30/30 [00:23<00:00,  1.29it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 298.30it/s]|
Column Shapes Score: 97.49%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 715.51it/s]|
Column Pair Trends Score: 99.56%

Overall Score (Average): 98.52%

##################################### Done #####################################
mse_c_mixup: 0.013391009162305291, aug_mse_c_mixup: 0.013281046191398227, delta_percent_c_mixup: -0.821170156589859, score_c_mixup: 0.9794719041076607
mse_ada: 0.013391009162305291, aug_mse_ada: 0.013453184061542185, delta_percent_ada: 0.4643033133896359, score_ada: 0.9862010876313321
mse_tabddpm: 0.013391009162305291, aug_mse_tabddpm: 0.01362458717047313, delta_percent_tabddpm: 1.7442898092052899, score_tabddpm: 0.9608103103761323
mse_ctgan: 0.013391009162305291, aug_mse_ctgan: 0.013879750664389094, delta_percent_ctgan: 3.649773487270653, score_ctgan: 0.9147786355773713
mse_tvae: 0.013391009162305291, aug_mse_tvae: 0.0133525494512

In [18]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/227_cpu_small.csv",
    results_dir="../experiments_all_baselines/227_cpu_small",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 227.76it/s]|
Column Shapes Score: 92.09%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 623.73it/s]|
Column Pair Trends Score: 99.92%

Overall Score (Average): 96.0%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 258.42it/s]|
Column Shapes Score: 94.22%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 621.37it/s]|
Column Pair Trends Score: 99.76%

Overall Score (Average): 96.99%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 378.72it/s]|
Column Shapes Score: 93.89%

(2/2) Evaluating Column Pair Trends: |█

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 182.78it/s]|
Column Shapes Score: 88.91%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 499.40it/s]|
Column Pair Trends Score: 93.92%

Overall Score (Average): 91.41%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 215.91it/s]|
Column Shapes Score: 92.17%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 544.12it/s]|
Column Pair Trends Score: 97.88%

Overall Score (Average): 95.03%

##################################### Running CRDA #####################################


Best trial: 18. Best value: 0.000600376: 100%|██████████| 30/30 [00:25<00:00,  1.16it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 370.29it/s]|
Column Shapes Score: 96.31%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 646.10it/s]|
Column Pair Trends Score: 99.68%

Overall Score (Average): 98.0%

##################################### Done #####################################
mse_c_mixup: 0.0006617519532631056, aug_mse_c_mixup: 0.0006740984736851157, delta_percent_c_mixup: 1.865732373154207, score_c_mixup: 0.9600265038130065
mse_ada: 0.0006617519532631056, aug_mse_ada: 0.0006746483893659393, delta_percent_ada: 1.948832344089226, score_ada: 0.9699180122929032
mse_tabddpm: 0.0006617519532631056, aug_mse_tabddpm: 0.0009400524826260054, delta_percent_tabddpm: 42.05511264312814, score_tabddpm: 0.9599391899882384
mse_ctgan: 0.0006617519532631056, aug_mse_ctgan: 0.0009456121888871292, delta_percent_ctgan: 42.89526222390518, score_ctgan: 0.9141419624800725
mse_tvae: 0.0006617519532631056, aug_mse_tvae: 0.0007856

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 232.97it/s]|
Column Shapes Score: 94.45%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 620.13it/s]|
Column Pair Trends Score: 99.51%

Overall Score (Average): 96.98%

##################################### Done #####################################
mse_c_mixup: 0.0006303828914319578, aug_mse_c_mixup: 0.0006340618922210354, delta_percent_c_mixup: 0.5836136797305046, score_c_mixup: 0.9630939768233527
mse_ada: 0.0006303828914319578, aug_mse_ada: 0.0006360952837939959, delta_percent_ada: 0.9061782037044198, score_ada: 0.9705945589641249
mse_tabddpm: 0.0006303828914319578, aug_mse_tabddpm: 0.0008417222890795512, delta_percent_tabddpm: 33.52556049982916, score_tabddpm: 0.9622950530422913
mse_ctgan: 0.0006303828914319578, aug_mse_ctgan: 0.0008476543342605592, delta_percent_ctgan: 34.46658305320667, score_ctgan: 0.9104180685809099
mse_tvae: 0.0006303828914319578, aug_mse_tvae: 0.0008

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 246.59it/s]|
Column Shapes Score: 88.62%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 608.21it/s]|
Column Pair Trends Score: 94.46%

Overall Score (Average): 91.54%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 205.14it/s]|
Column Shapes Score: 91.29%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 532.25it/s]|
Column Pair Trends Score: 98.04%

Overall Score (Average): 94.66%

##################################### Running CRDA #####################################


Best trial: 15. Best value: 0.000591343: 100%|██████████| 30/30 [00:26<00:00,  1.12it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 211.52it/s]|
Column Shapes Score: 96.54%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 569.97it/s]|
Column Pair Trends Score: 99.84%

Overall Score (Average): 98.19%

##################################### Done #####################################
mse_c_mixup: 0.0006585615120750768, aug_mse_c_mixup: 0.0006520769830410708, delta_percent_c_mixup: -0.9846504715366319, score_c_mixup: 0.9688361386467639
mse_ada: 0.0006585615120750768, aug_mse_ada: 0.0006571126053326099, delta_percent_ada: -0.22001084422644843, score_ada: 0.9707953414755868
mse_tabddpm: 0.0006585615120750768, aug_mse_tabddpm: 0.0009679242514788207, delta_percent_tabddpm: 46.97552677030361, score_tabddpm: 0.960265919161652
mse_ctgan: 0.0006585615120750768, aug_mse_ctgan: 0.0008677862630681601, delta_percent_ctgan: 31.769963345387765, score_ctgan: 0.9153820554152164
mse_tvae: 0.0006585615120750768, aug_mse_tvae: 0.0

In [19]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/294_satellite_image.csv",
    results_dir="../experiments_all_baselines/294_satellite_image",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 274.26it/s]|
Column Shapes Score: 94.41%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 734.16it/s]|
Column Pair Trends Score: 99.87%

Overall Score (Average): 97.14%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 763.88it/s]|
Column Shapes Score: 98.14%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 754.04it/s]|
Column Pair Trends Score: 99.78%

Overall Score (Average): 98.96%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 359.93it/s]|
Column Shapes Score: 86.33%

(2/2) Evaluating Column Pair Trend

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 263.70it/s]|
Column Shapes Score: 93.32%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 781.12it/s]|
Column Pair Trends Score: 87.1%

Overall Score (Average): 90.21%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 316.63it/s]|
Column Shapes Score: 96.31%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:01<00:00, 592.58it/s]|
Column Pair Trends Score: 98.82%

Overall Score (Average): 97.56%

##################################### Running CRDA #####################################


Best trial: 8. Best value: 0.0106275: 100%|██████████| 30/30 [00:30<00:00,  1.01s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 359.61it/s]|
Column Shapes Score: 99.18%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 792.27it/s]|
Column Pair Trends Score: 99.74%

Overall Score (Average): 99.46%

##################################### Done #####################################
mse_c_mixup: 0.014590294631114236, aug_mse_c_mixup: 0.016118462988801863, delta_percent_c_mixup: 10.473869077522007, score_c_mixup: 0.9713742828918305
mse_ada: 0.014590294631114236, aug_mse_ada: 0.014427087492173913, delta_percent_ada: -1.1186007072967465, score_ada: 0.9896055380550232
mse_tabddpm: 0.014590294631114236, aug_mse_tabddpm: 0.015018452787903015, delta_percent_tabddpm: 2.934540854820845, score_tabddpm: 0.9009934152167658
mse_ctgan: 0.014590294631114236, aug_mse_ctgan: 0.015643155284379946, delta_percent_ctgan: 7.216171296640256, score_ctgan: 0.9021174628669038
mse_tvae: 0.014590294631114236, aug_mse_tvae: 0.0165558678

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 171.01it/s]|
Column Shapes Score: 99.26%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 792.31it/s]|
Column Pair Trends Score: 99.75%

Overall Score (Average): 99.51%

##################################### Done #####################################
mse_c_mixup: 0.013882660670265466, aug_mse_c_mixup: 0.014969001261581405, delta_percent_c_mixup: 7.8251613081829055, score_c_mixup: 0.9785595289072981
mse_ada: 0.013882660670265466, aug_mse_ada: 0.014592006136669092, delta_percent_ada: 5.109578655357725, score_ada: 0.9886346993414756
mse_tabddpm: 0.013882660670265466, aug_mse_tabddpm: 0.014778658471199283, delta_percent_tabddpm: 6.4540783803273865, score_tabddpm: 0.9014718183525274
mse_ctgan: 0.013882660670265466, aug_mse_ctgan: 0.015503375922570228, delta_percent_ctgan: 11.674384981375265, score_ctgan: 0.8999861872497869
mse_tvae: 0.013882660670265466, aug_mse_tvae: 0.0159011214

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 292.32it/s]|
Column Shapes Score: 92.89%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 795.33it/s]|
Column Pair Trends Score: 86.6%

Overall Score (Average): 89.74%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 287.07it/s]|
Column Shapes Score: 96.0%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 774.31it/s]|
Column Pair Trends Score: 98.05%

Overall Score (Average): 97.02%

##################################### Running CRDA #####################################


Best trial: 2. Best value: 0.0105766: 100%|██████████| 30/30 [00:30<00:00,  1.03s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 363.31it/s]|
Column Shapes Score: 98.76%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 799.43it/s]|
Column Pair Trends Score: 99.45%

Overall Score (Average): 99.11%

##################################### Done #####################################
mse_c_mixup: 0.012222237566649294, aug_mse_c_mixup: 0.012299947209023658, delta_percent_c_mixup: 0.635805366657328, score_c_mixup: 0.9841237961701559
mse_ada: 0.012222237566649294, aug_mse_ada: 0.0125990129784685, delta_percent_ada: 3.0827040446939935, score_ada: 0.98975488881812
mse_tabddpm: 0.012222237566649294, aug_mse_tabddpm: 0.013582817239233349, delta_percent_tabddpm: 11.132001527254356, score_tabddpm: 0.9005230319935555
mse_ctgan: 0.012222237566649294, aug_mse_ctgan: 0.013307261757255573, delta_percent_ctgan: 8.877459505180745, score_ctgan: 0.8974374966005152
mse_tvae: 0.012222237566649294, aug_mse_tvae: 0.013269824398732

In [20]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/503_wind.csv",
    results_dir="../experiments_all_baselines/503_wind",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 261.17it/s]|
Column Shapes Score: 95.1%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 695.14it/s]|
Column Pair Trends Score: 99.93%

Overall Score (Average): 97.52%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 498.74it/s]|
Column Shapes Score: 98.8%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 687.02it/s]|
Column Pair Trends Score: 99.74%

Overall Score (Average): 99.27%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 530.79it/s]|
Column Shapes Score: 98.02%

(2/2) Evaluating Column Pair Trends:

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 184.78it/s]|
Column Shapes Score: 89.72%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 656.27it/s]|
Column Pair Trends Score: 85.87%

Overall Score (Average): 87.79%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 371.23it/s]|
Column Shapes Score: 94.12%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 658.46it/s]|
Column Pair Trends Score: 94.77%

Overall Score (Average): 94.45%

##################################### Running CRDA #####################################


Best trial: 21. Best value: 0.00462284: 100%|██████████| 30/30 [00:27<00:00,  1.11it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 207.11it/s]|
Column Shapes Score: 99.05%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 757.22it/s]|
Column Pair Trends Score: 99.29%

Overall Score (Average): 99.17%

##################################### Done #####################################
mse_c_mixup: 0.004967977097783902, aug_mse_c_mixup: 0.004875585200483012, delta_percent_c_mixup: -1.8597488571777743, score_c_mixup: 0.97516730572786
mse_ada: 0.004967977097783902, aug_mse_ada: 0.004991058400416109, delta_percent_ada: 0.464601631164986, score_ada: 0.9926986557163885
mse_tabddpm: 0.004967977097783902, aug_mse_tabddpm: 0.004880496322594368, delta_percent_tabddpm: -1.7608932864959712, score_tabddpm: 0.9777380388615982
mse_ctgan: 0.004967977097783902, aug_mse_ctgan: 0.005358629096085177, delta_percent_ctgan: 7.863401755123554, score_ctgan: 0.8779492921077796
mse_tvae: 0.004967977097783902, aug_mse_tvae: 0.00535359537

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 249.28it/s]|
Column Shapes Score: 98.95%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 647.57it/s]|
Column Pair Trends Score: 99.42%

Overall Score (Average): 99.18%

##################################### Done #####################################
mse_c_mixup: 0.005756158996860011, aug_mse_c_mixup: 0.005696837877745665, delta_percent_c_mixup: -1.0305677648359939, score_c_mixup: 0.9801265774639736
mse_ada: 0.005756158996860011, aug_mse_ada: 0.005686928793168867, delta_percent_ada: -1.2027152781726307, score_ada: 0.9913681289551783
mse_tabddpm: 0.005756158996860011, aug_mse_tabddpm: 0.005719712214465563, delta_percent_tabddpm: -0.6331788683100912, score_tabddpm: 0.9799593354544662
mse_ctgan: 0.005756158996860011, aug_mse_ctgan: 0.00607392634058299, delta_percent_ctgan: 5.520475440242725, score_ctgan: 0.8777213562476716
mse_tvae: 0.005756158996860011, aug_mse_tvae: 0.00603416

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 234.86it/s]|
Column Shapes Score: 92.44%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 651.77it/s]|
Column Pair Trends Score: 85.35%

Overall Score (Average): 88.9%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 316.74it/s]|
Column Shapes Score: 93.23%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 645.15it/s]|
Column Pair Trends Score: 96.45%

Overall Score (Average): 94.84%

##################################### Running CRDA #####################################


Best trial: 1. Best value: 0.00427855: 100%|██████████| 30/30 [00:28<00:00,  1.05it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 387.18it/s]|
Column Shapes Score: 99.48%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 693.65it/s]|
Column Pair Trends Score: 99.75%

Overall Score (Average): 99.62%

##################################### Done #####################################
mse_c_mixup: 0.005515268959175753, aug_mse_c_mixup: 0.005252742067531026, delta_percent_c_mixup: -4.76000161710991, score_c_mixup: 0.9883292594307289
mse_ada: 0.005515268959175753, aug_mse_ada: 0.005453631858993323, delta_percent_ada: -1.1175719740718033, score_ada: 0.99316111198715
mse_tabddpm: 0.005515268959175753, aug_mse_tabddpm: 0.0054076096616645745, delta_percent_tabddpm: -1.952022617719588, score_tabddpm: 0.9790463231088677
mse_ctgan: 0.005515268959175753, aug_mse_ctgan: 0.005751770673808699, delta_percent_ctgan: 4.288126587906069, score_ctgan: 0.8889835924335496
mse_tvae: 0.005515268959175753, aug_mse_tvae: 0.00546694896

In [21]:
config = Config(
    baseline="xgboost",
    dataset_path="../data/ParkinsonsTelemonitoring.csv",
    results_dir="../experiments_all_baselines/ParkinsonsTelemonitoring",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 43.55it/s]|
Column Shapes Score: 94.41%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 652.49it/s]|
Column Pair Trends Score: 99.92%

Overall Score (Average): 97.17%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 133.89it/s]|
Column Shapes Score: 98.17%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 602.46it/s]|
Column Pair Trends Score: 99.73%

Overall Score (Average): 98.95%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 68.96it/s]|
Column Shapes Score: 96.2%

(2/2) Evaluating Column Pair Trends: 

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 28.74it/s]|
Column Shapes Score: 91.25%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 271.76it/s]|
Column Pair Trends Score: 88.9%

Overall Score (Average): 90.07%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 30.19it/s]|
Column Shapes Score: 93.34%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 624.15it/s]|
Column Pair Trends Score: 96.85%

Overall Score (Average): 95.09%

##################################### Running CRDA #####################################


Best trial: 16. Best value: 0.000100535: 100%|██████████| 30/30 [00:34<00:00,  1.16s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 354.36it/s]|
Column Shapes Score: 99.56%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 524.74it/s]|
Column Pair Trends Score: 99.86%

Overall Score (Average): 99.71%

##################################### Done #####################################
mse_c_mixup: 8.859484231095864e-05, aug_mse_c_mixup: 0.00025053842509014426, delta_percent_c_mixup: 182.79120833104548, score_c_mixup: 0.9716747183935901
mse_ada: 8.859484231095864e-05, aug_mse_ada: 0.00019541454017455231, delta_percent_ada: 120.57101189781194, score_ada: 0.9894776879512417
mse_tabddpm: 8.859484231095864e-05, aug_mse_tabddpm: 0.00039251916447463804, delta_percent_tabddpm: 343.0496790060722, score_tabddpm: 0.9484508048814067
mse_ctgan: 8.859484231095864e-05, aug_mse_ctgan: 0.0008165134989852013, delta_percent_ctgan: 821.6264487715033, score_ctgan: 0.9007267656402931
mse_tvae: 8.859484231095864e-05, aug_mse_tvae: 0

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 495.33it/s]|
Column Shapes Score: 99.8%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 702.96it/s]|
Column Pair Trends Score: 99.97%

Overall Score (Average): 99.88%

##################################### Done #####################################
mse_c_mixup: 0.00013533256333917834, aug_mse_c_mixup: 0.0002360442174499439, delta_percent_c_mixup: 74.41790181595553, score_c_mixup: 0.9785537311155323
mse_ada: 0.00013533256333917834, aug_mse_ada: 0.00022006258746035739, delta_percent_ada: 62.60874842725305, score_ada: 0.9908069023327051
mse_tabddpm: 0.00013533256333917834, aug_mse_tabddpm: 0.0003599005214984401, delta_percent_tabddpm: 165.93785901803727, score_tabddpm: 0.9453384628900237
mse_ctgan: 0.00013533256333917834, aug_mse_ctgan: 0.0008378876920774709, delta_percent_ctgan: 519.1323591333359, score_ctgan: 0.9003538462173228
mse_tvae: 0.00013533256333917834, aug_mse_tvae: 

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:01<00:00, 19.85it/s]|
Column Shapes Score: 87.65%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 664.70it/s]|
Column Pair Trends Score: 89.83%

Overall Score (Average): 88.74%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 26.92it/s]|
Column Shapes Score: 90.4%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 676.78it/s]|
Column Pair Trends Score: 97.19%

Overall Score (Average): 93.8%

##################################### Running CRDA #####################################


Best trial: 20. Best value: 0.000111757: 100%|██████████| 30/30 [00:36<00:00,  1.23s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 264.38it/s]|
Column Shapes Score: 99.81%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 401.19it/s]|
Column Pair Trends Score: 99.98%

Overall Score (Average): 99.89%

##################################### Done #####################################
mse_c_mixup: 0.00010126160825737233, aug_mse_c_mixup: 0.0002064109899179815, delta_percent_c_mixup: 103.8393360229431, score_c_mixup: 0.9838586465834186
mse_ada: 0.00010126160825737233, aug_mse_ada: 0.0002262442588572025, delta_percent_ada: 123.42550424655225, score_ada: 0.9882748734745106
mse_tabddpm: 0.00010126160825737233, aug_mse_tabddpm: 0.00045703681843337946, delta_percent_tabddpm: 351.3426423879704, score_tabddpm: 0.9486928435897863
mse_ctgan: 0.00010126160825737233, aug_mse_ctgan: 0.0006622975952219728, delta_percent_ctgan: 554.046095671954, score_ctgan: 0.8873753056234048
mse_tvae: 0.00010126160825737233, aug_mse_tvae: 

In [23]:
config = Config(
    baseline="mlp",
    dataset_path="../data/HousePrice.csv",
    results_dir="../experiments_all_baselines/HousePrice",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 765.54it/s]|
Column Shapes Score: 92.43%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 733.88it/s]|
Column Pair Trends Score: 99.74%

Overall Score (Average): 96.08%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1082.09it/s]|
Column Shapes Score: 95.61%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 789.11it/s]|
Column Pair Trends Score: 99.16%

Overall Score (Average): 97.38%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 788.64it/s]|
Column Shapes Score: 92.15%

(2/2) Evaluating Column Pair Trends: |█████

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 597.60it/s]|
Column Shapes Score: 88.21%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 739.02it/s]|
Column Pair Trends Score: 96.95%

Overall Score (Average): 92.58%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 341.76it/s]|
Column Shapes Score: 76.31%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 752.12it/s]|
Column Pair Trends Score: 92.18%

Overall Score (Average): 84.25%

##################################### Running CRDA #####################################


Best trial: 27. Best value: 0.000384797: 100%|██████████| 30/30 [00:24<00:00,  1.20it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1439.12it/s]|
Column Shapes Score: 98.25%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 787.29it/s]|
Column Pair Trends Score: 99.57%

Overall Score (Average): 98.91%

##################################### Done #####################################
mse_c_mixup: 0.0008274401745728672, aug_mse_c_mixup: 0.00036473878152515923, delta_percent_c_mixup: -55.919618996812545, score_c_mixup: 0.9608461009348246
mse_ada: 0.0008274401745728672, aug_mse_ada: 0.0003388097892759738, delta_percent_ada: -59.05325850888607, score_ada: 0.9738293331276202
mse_tabddpm: 0.0008274401745728672, aug_mse_tabddpm: 0.0008758476031676569, delta_percent_tabddpm: 5.8502632676468895, score_tabddpm: 0.9544816933269744
mse_ctgan: 0.0008274401745728672, aug_mse_ctgan: 0.014055432754412654, delta_percent_ctgan: 1598.66453023848, score_ctgan: 0.9258214988437202
mse_tvae: 0.0008274401745728672, aug_mse_tvae: 0.0079

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1294.34it/s]|
Column Shapes Score: 98.36%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 685.99it/s]|
Column Pair Trends Score: 99.93%

Overall Score (Average): 99.14%

##################################### Done #####################################
mse_c_mixup: 0.000983218386366038, aug_mse_c_mixup: 0.0004159870230692543, delta_percent_c_mixup: -57.69128925601802, score_c_mixup: 0.9612883618706087
mse_ada: 0.000983218386366038, aug_mse_ada: 0.00033010491931179, delta_percent_ada: -66.42608357520109, score_ada: 0.9758239738429584
mse_tabddpm: 0.000983218386366038, aug_mse_tabddpm: 0.0005267624860626059, delta_percent_tabddpm: -46.42467092082024, score_tabddpm: 0.9576847205963352
mse_ctgan: 0.000983218386366038, aug_mse_ctgan: 0.025292090116326982, delta_percent_ctgan: 2472.3776596373677, score_ctgan: 0.9270522759888955
mse_tvae: 0.000983218386366038, aug_mse_tvae: 0.00283072350

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 537.84it/s]|
Column Shapes Score: 86.9%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 677.82it/s]|
Column Pair Trends Score: 96.85%

Overall Score (Average): 91.87%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 324.36it/s]|
Column Shapes Score: 75.29%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 752.19it/s]|
Column Pair Trends Score: 91.22%

Overall Score (Average): 83.25%

##################################### Running CRDA #####################################


Best trial: 26. Best value: 0.000297052: 100%|██████████| 30/30 [00:25<00:00,  1.20it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 1160.93it/s]|
Column Shapes Score: 97.61%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 649.28it/s]|
Column Pair Trends Score: 99.71%

Overall Score (Average): 98.66%

##################################### Done #####################################
mse_c_mixup: 0.0013288103412194194, aug_mse_c_mixup: 0.0006770018919518442, delta_percent_c_mixup: -49.05203015423745, score_c_mixup: 0.9658692354594554
mse_ada: 0.0013288103412194194, aug_mse_ada: 0.00047380747571558885, delta_percent_ada: -64.34348371485532, score_ada: 0.9737386101693886
mse_tabddpm: 0.0013288103412194194, aug_mse_tabddpm: 0.0006249675446227006, delta_percent_tabddpm: -52.96788975549499, score_tabddpm: 0.9541274317795663
mse_ctgan: 0.0013288103412194194, aug_mse_ctgan: 0.028201830172879038, delta_percent_ctgan: 2022.3367472440689, score_ctgan: 0.9187218694610307
mse_tvae: 0.0013288103412194194, aug_mse_tvae: 0.011

In [24]:
config = Config(
    baseline="mlp",
    dataset_path="../data/623_fri_c4_1000_10.csv",
    results_dir="../experiments_all_baselines/623_fri_c4_1000_10",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1025.52it/s]|
Column Shapes Score: 93.91%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 851.09it/s]|
Column Pair Trends Score: 99.71%

Overall Score (Average): 96.81%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1765.35it/s]|
Column Shapes Score: 98.32%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 855.26it/s]|
Column Pair Trends Score: 99.23%

Overall Score (Average): 98.78%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1512.60it/s]|
Column Shapes Score: 97.77%

(2/2) Evaluating Column Pair Trends

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 581.33it/s]|
Column Shapes Score: 88.86%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 731.99it/s]|
Column Pair Trends Score: 94.91%

Overall Score (Average): 91.88%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 386.02it/s]|
Column Shapes Score: 79.8%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 873.13it/s]|
Column Pair Trends Score: 93.58%

Overall Score (Average): 86.69%

##################################### Running CRDA #####################################


Best trial: 9. Best value: 0.00206168: 100%|██████████| 30/30 [00:24<00:00,  1.22it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1545.38it/s]|
Column Shapes Score: 98.39%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 837.82it/s]|
Column Pair Trends Score: 99.42%

Overall Score (Average): 98.9%

##################################### Done #####################################
mse_c_mixup: 0.003384473117197005, aug_mse_c_mixup: 0.008731912514130887, delta_percent_c_mixup: 157.99916890350693, score_c_mixup: 0.9681163199378804
mse_ada: 0.003384473117197005, aug_mse_ada: 0.005507873820372136, delta_percent_ada: 62.739476120694235, score_ada: 0.9877836763287406
mse_tabddpm: 0.003384473117197005, aug_mse_tabddpm: 0.003067634053251368, delta_percent_tabddpm: -9.361547661162719, score_tabddpm: 0.9828355709602781
mse_ctgan: 0.003384473117197005, aug_mse_ctgan: 0.00865359412506326, delta_percent_ctgan: 155.685119231501, score_ctgan: 0.9188313296074868
mse_tvae: 0.003384473117197005, aug_mse_tvae: 0.00562184528946

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1774.17it/s]|
Column Shapes Score: 98.99%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 804.97it/s]|
Column Pair Trends Score: 99.61%

Overall Score (Average): 99.3%

##################################### Done #####################################
mse_c_mixup: 0.00297968758025445, aug_mse_c_mixup: 0.006417511032794282, delta_percent_c_mixup: 115.37529891795768, score_c_mixup: 0.975549092771895
mse_ada: 0.00297968758025445, aug_mse_ada: 0.0038052259545163923, delta_percent_ada: 27.705534624923523, score_ada: 0.988141625226594
mse_tabddpm: 0.00297968758025445, aug_mse_tabddpm: 0.002682687106527898, delta_percent_tabddpm: -9.967503831431534, score_tabddpm: 0.9835565747849175
mse_ctgan: 0.00297968758025445, aug_mse_ctgan: 0.011301448075085263, delta_percent_ctgan: 279.282987584899, score_ctgan: 0.9214715170664041
mse_tvae: 0.00297968758025445, aug_mse_tvae: 0.004404570162190644,

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 429.59it/s]|
Column Shapes Score: 87.59%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 795.08it/s]|
Column Pair Trends Score: 93.87%

Overall Score (Average): 90.73%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 372.30it/s]|
Column Shapes Score: 78.95%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 880.28it/s]|
Column Pair Trends Score: 93.43%

Overall Score (Average): 86.19%

##################################### Running CRDA #####################################


Best trial: 3. Best value: 0.00297522: 100%|██████████| 30/30 [00:25<00:00,  1.20it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1575.08it/s]|
Column Shapes Score: 99.14%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 786.71it/s]|
Column Pair Trends Score: 99.73%

Overall Score (Average): 99.43%

##################################### Done #####################################
mse_c_mixup: 0.00403551505201902, aug_mse_c_mixup: 0.005149260790086274, delta_percent_c_mixup: 27.598602005214502, score_c_mixup: 0.9808516961969251
mse_ada: 0.00403551505201902, aug_mse_ada: 0.004231977877854637, delta_percent_ada: 4.868345757682745, score_ada: 0.9863672034398263
mse_tabddpm: 0.00403551505201902, aug_mse_tabddpm: 0.0030734811034282814, delta_percent_tabddpm: -23.839186230006025, score_tabddpm: 0.9854782182194162
mse_ctgan: 0.00403551505201902, aug_mse_ctgan: 0.006745088242486754, delta_percent_ctgan: 67.14318136695093, score_ctgan: 0.9072884327635965
mse_tvae: 0.00403551505201902, aug_mse_tvae: 0.003819227160803

In [25]:
config = Config(
    baseline="mlp",
    dataset_path="../data/ConcreteCompressiveStrength.csv",
    results_dir="../experiments_all_baselines/ConcreteCompressiveStrength",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 807.39it/s]|
Column Shapes Score: 92.01%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 769.11it/s]|
Column Pair Trends Score: 99.71%

Overall Score (Average): 95.86%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 998.86it/s]|
Column Shapes Score: 94.15%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 815.28it/s]|
Column Pair Trends Score: 99.44%

Overall Score (Average): 96.79%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 612.95it/s]|
Column Shapes Score: 87.85%

(2/2) Evaluating Column Pair Trends: |██████

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 602.04it/s]|
Column Shapes Score: 88.49%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 817.27it/s]|
Column Pair Trends Score: 94.3%

Overall Score (Average): 91.39%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 295.05it/s]|
Column Shapes Score: 78.49%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 782.32it/s]|
Column Pair Trends Score: 93.25%

Overall Score (Average): 85.87%

##################################### Running CRDA #####################################


Best trial: 21. Best value: 0.00234357: 100%|██████████| 30/30 [00:20<00:00,  1.48it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1731.75it/s]|
Column Shapes Score: 98.86%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 763.76it/s]|
Column Pair Trends Score: 99.75%

Overall Score (Average): 99.31%

##################################### Done #####################################
mse_c_mixup: 0.00572893406774741, aug_mse_c_mixup: 0.005417557097725888, delta_percent_c_mixup: -5.435164139425221, score_c_mixup: 0.9585734610400833
mse_ada: 0.00572893406774741, aug_mse_ada: 0.005719873004656575, delta_percent_ada: -0.15816315886486926, score_ada: 0.967929247802235
mse_tabddpm: 0.00572893406774741, aug_mse_tabddpm: 0.004700410060037888, delta_percent_tabddpm: -17.953147925019366, score_tabddpm: 0.9343196301958034
mse_ctgan: 0.00572893406774741, aug_mse_ctgan: 0.010317513413030256, delta_percent_ctgan: 80.09481853030043, score_ctgan: 0.9139342331470455
mse_tvae: 0.00572893406774741, aug_mse_tvae: 0.0088611168668460

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1265.97it/s]|
Column Shapes Score: 97.42%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 794.72it/s]|
Column Pair Trends Score: 99.43%

Overall Score (Average): 98.42%

##################################### Done #####################################
mse_c_mixup: 0.004656082213688997, aug_mse_c_mixup: 0.0044922263799412054, delta_percent_c_mixup: -3.5191782753760483, score_c_mixup: 0.9583664774246368
mse_ada: 0.004656082213688997, aug_mse_ada: 0.004646357888618527, delta_percent_ada: -0.20885209118258127, score_ada: 0.9711348858353632
mse_tabddpm: 0.004656082213688997, aug_mse_tabddpm: 0.004522644843752241, delta_percent_tabddpm: -2.86587228946359, score_tabddpm: 0.9367214412446252
mse_ctgan: 0.004656082213688997, aug_mse_ctgan: 0.009557415196197306, delta_percent_ctgan: 105.26732041153117, score_ctgan: 0.8951310236166317
mse_tvae: 0.004656082213688997, aug_mse_tvae: 0.008530270

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 487.12it/s]|
Column Shapes Score: 84.58%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 789.23it/s]|
Column Pair Trends Score: 94.18%

Overall Score (Average): 89.38%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 406.49it/s]|
Column Shapes Score: 80.56%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 866.30it/s]|
Column Pair Trends Score: 95.1%

Overall Score (Average): 87.83%

##################################### Running CRDA #####################################


Best trial: 10. Best value: 0.00381433: 100%|██████████| 30/30 [00:19<00:00,  1.52it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 1056.09it/s]|
Column Shapes Score: 95.36%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 761.65it/s]|
Column Pair Trends Score: 99.55%

Overall Score (Average): 97.45%

##################################### Done #####################################
mse_c_mixup: 0.00497442369739306, aug_mse_c_mixup: 0.00456011465982136, delta_percent_c_mixup: -8.328784654769684, score_c_mixup: 0.9640390116087175
mse_ada: 0.00497442369739306, aug_mse_ada: 0.004988295946868048, delta_percent_ada: 0.27887148982217574, score_ada: 0.9646830130436554
mse_tabddpm: 0.00497442369739306, aug_mse_tabddpm: 0.005109884909689248, delta_percent_tabddpm: 2.723153887498133, score_tabddpm: 0.9355520956389636
mse_ctgan: 0.00497442369739306, aug_mse_ctgan: 0.013297596093875265, delta_percent_ctgan: 167.3193298923073, score_ctgan: 0.8937718194725595
mse_tvae: 0.00497442369739306, aug_mse_tvae: 0.005090066138949224,

In [26]:
config = Config(
    baseline="mlp",
    dataset_path="../data/EnergyEfficiency.csv",
    results_dir="../experiments_all_baselines/EnergyEfficiency",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1117.44it/s]|
Column Shapes Score: 90.86%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 888.98it/s]|
Column Pair Trends Score: 99.84%

Overall Score (Average): 95.35%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1454.13it/s]|
Column Shapes Score: 95.0%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 855.65it/s]|
Column Pair Trends Score: 99.5%

Overall Score (Average): 97.25%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1056.74it/s]|
Column Shapes Score: 90.67%

(2/2) Evaluating Column Pair Trends: 

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1044.74it/s]|
Column Shapes Score: 89.85%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 822.34it/s]|
Column Pair Trends Score: 91.5%

Overall Score (Average): 90.68%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 870.71it/s]|
Column Shapes Score: 86.28%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 858.32it/s]|
Column Pair Trends Score: 94.84%

Overall Score (Average): 90.56%

##################################### Running CRDA #####################################


Best trial: 22. Best value: 0.000727606: 100%|██████████| 30/30 [00:18<00:00,  1.64it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1578.53it/s]|
Column Shapes Score: 96.53%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 826.05it/s]|
Column Pair Trends Score: 99.84%

Overall Score (Average): 98.18%

##################################### Done #####################################
mse_c_mixup: 0.0013208531242431079, aug_mse_c_mixup: 0.001119209621771587, delta_percent_c_mixup: -15.266156302357173, score_c_mixup: 0.9534935079616078
mse_ada: 0.0013208531242431079, aug_mse_ada: 0.0013274362542828595, delta_percent_ada: 0.49839985377056933, score_ada: 0.9725072254448313
mse_tabddpm: 0.0013208531242431079, aug_mse_tabddpm: 0.001574061188738875, delta_percent_tabddpm: 19.170039412282403, score_tabddpm: 0.9503396726367552
mse_ctgan: 0.0013208531242431079, aug_mse_ctgan: 0.007126346829607597, delta_percent_ctgan: 439.5260607564696, score_ctgan: 0.9067531306402176
mse_tvae: 0.0013208531242431079, aug_mse_tvae: 0.004

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 2162.12it/s]|
Column Shapes Score: 99.46%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 858.22it/s]|
Column Pair Trends Score: 99.78%

Overall Score (Average): 99.62%

##################################### Done #####################################
mse_c_mixup: 0.0015495996238625196, aug_mse_c_mixup: 0.0014396753254955527, delta_percent_c_mixup: -7.093722576736984, score_c_mixup: 0.9550315108122129
mse_ada: 0.0015495996238625196, aug_mse_ada: 0.0010488466741672892, delta_percent_ada: -32.31498910970678, score_ada: 0.974147113360465
mse_tabddpm: 0.0015495996238625196, aug_mse_tabddpm: 0.00153196987214018, delta_percent_tabddpm: -1.137697212289965, score_tabddpm: 0.9443573217316454
mse_ctgan: 0.0015495996238625196, aug_mse_ctgan: 0.004625694683864421, delta_percent_ctgan: 198.50902211336708, score_ctgan: 0.8828790291104766
mse_tvae: 0.0015495996238625196, aug_mse_tvae: 0.00151

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 871.16it/s]|
Column Shapes Score: 87.21%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 808.90it/s]|
Column Pair Trends Score: 92.05%

Overall Score (Average): 89.63%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 684.83it/s]|
Column Shapes Score: 83.01%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 179.08it/s]|
Column Pair Trends Score: 93.98%

Overall Score (Average): 88.5%

##################################### Running CRDA #####################################


Best trial: 27. Best value: 0.000867901: 100%|██████████| 30/30 [00:16<00:00,  1.77it/s]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 1805.71it/s]|
Column Shapes Score: 97.66%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 853.92it/s]|
Column Pair Trends Score: 99.76%

Overall Score (Average): 98.71%

##################################### Done #####################################
mse_c_mixup: 0.001426637979383012, aug_mse_c_mixup: 0.0010895033879573717, delta_percent_c_mixup: -23.631404483668895, score_c_mixup: 0.9597379013211836
mse_ada: 0.001426637979383012, aug_mse_ada: 0.0010395782843268166, delta_percent_ada: -27.130898002840905, score_ada: 0.9714104945082972
mse_tabddpm: 0.001426637979383012, aug_mse_tabddpm: 0.0014848659157331626, delta_percent_tabddpm: 4.081479477739177, score_tabddpm: 0.9524612543688484
mse_ctgan: 0.001426637979383012, aug_mse_ctgan: 0.004896807552493761, delta_percent_ctgan: 243.24107610057578, score_ctgan: 0.8962930053056223
mse_tvae: 0.001426637979383012, aug_mse_tvae: 0.003295

In [27]:
config = Config(
    baseline="mlp",
    dataset_path="../data/WineQuality.csv",
    results_dir="../experiments_all_baselines/WineQuality",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 52.98it/s]|
Column Shapes Score: 94.18%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 734.70it/s]|
Column Pair Trends Score: 99.9%

Overall Score (Average): 97.04%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 127.86it/s]|
Column Shapes Score: 97.59%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 736.76it/s]|
Column Pair Trends Score: 99.67%

Overall Score (Average): 98.63%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 93.89it/s]|
Column Shapes Score: 96.64%

(2/2) Evaluating Column Pair Trends: |███

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 30.53it/s]|
Column Shapes Score: 89.56%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 696.96it/s]|
Column Pair Trends Score: 93.88%

Overall Score (Average): 91.72%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 29.04it/s]|
Column Shapes Score: 89.76%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 733.64it/s]|
Column Pair Trends Score: 97.88%

Overall Score (Average): 93.82%

##################################### Running CRDA #####################################


Best trial: 23. Best value: 0.0135307: 100%|██████████| 30/30 [01:06<00:00,  2.23s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 238.22it/s]|
Column Shapes Score: 98.45%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 769.13it/s]|
Column Pair Trends Score: 99.75%

Overall Score (Average): 99.1%

##################################### Done #####################################
mse_c_mixup: 0.015638992614758873, aug_mse_c_mixup: 0.015607872072490828, delta_percent_c_mixup: -0.1989932666038626, score_c_mixup: 0.9703916627646135
mse_ada: 0.015638992614758873, aug_mse_ada: 0.019127658006462836, delta_percent_ada: 22.30748154719141, score_ada: 0.9862893182770416
mse_tabddpm: 0.015638992614758873, aug_mse_tabddpm: 0.014297229293867874, delta_percent_tabddpm: -8.579601985518853, score_tabddpm: 0.9607003545486499
mse_ctgan: 0.015638992614758873, aug_mse_ctgan: 0.015345830727887033, delta_percent_ctgan: -1.874557358606183, score_ctgan: 0.9172227042721945
mse_tvae: 0.015638992614758873, aug_mse_tvae: 0.015355551818

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 249.63it/s]|
Column Shapes Score: 98.9%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 345.18it/s]|
Column Pair Trends Score: 99.76%

Overall Score (Average): 99.33%

##################################### Done #####################################
mse_c_mixup: 0.014166669167379308, aug_mse_c_mixup: 0.014868639450045252, delta_percent_c_mixup: 4.955083473554438, score_c_mixup: 0.9749381733579676
mse_ada: 0.014166669167379308, aug_mse_ada: 0.016882284203244, delta_percent_ada: 19.169043928249312, score_ada: 0.9862349233911314
mse_tabddpm: 0.014166669167379308, aug_mse_tabddpm: 0.013040387273238883, delta_percent_tabddpm: -7.950223731728292, score_tabddpm: 0.9595373372644138
mse_ctgan: 0.014166669167379308, aug_mse_ctgan: 0.014510522602562545, delta_percent_ctgan: 2.427200290488932, score_ctgan: 0.8982832602407893
mse_tvae: 0.014166669167379308, aug_mse_tvae: 0.01358646813689922

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 28.20it/s]|
Column Shapes Score: 88.88%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 753.31it/s]|
Column Pair Trends Score: 94.07%

Overall Score (Average): 91.48%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 32.71it/s]|
Column Shapes Score: 90.3%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 761.18it/s]|
Column Pair Trends Score: 98.04%

Overall Score (Average): 94.17%

##################################### Running CRDA #####################################


Best trial: 29. Best value: 0.0136257: 100%|██████████| 30/30 [01:09<00:00,  2.33s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 328.69it/s]|
Column Shapes Score: 99.02%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 767.52it/s]|
Column Pair Trends Score: 99.83%

Overall Score (Average): 99.42%

##################################### Done #####################################
mse_c_mixup: 0.014382218398251314, aug_mse_c_mixup: 0.017816571133356635, delta_percent_c_mixup: 23.87915855542071, score_c_mixup: 0.9794719041076607
mse_ada: 0.014382218398251314, aug_mse_ada: 0.01783565819233041, delta_percent_ada: 24.011871454399465, score_ada: 0.9862010876313321
mse_tabddpm: 0.014382218398251314, aug_mse_tabddpm: 0.013740687469872621, delta_percent_tabddpm: -4.46058396983246, score_tabddpm: 0.9608103103761323
mse_ctgan: 0.014382218398251314, aug_mse_ctgan: 0.01459929011657825, delta_percent_ctgan: 1.509306230208049, score_ctgan: 0.9147786355773713
mse_tvae: 0.014382218398251314, aug_mse_tvae: 0.0141643395278158

In [28]:
config = Config(
    baseline="mlp",
    dataset_path="../data/227_cpu_small.csv",
    results_dir="../experiments_all_baselines/227_cpu_small",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 211.38it/s]|
Column Shapes Score: 92.09%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 595.96it/s]|
Column Pair Trends Score: 99.92%

Overall Score (Average): 96.0%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 245.89it/s]|
Column Shapes Score: 94.22%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 625.32it/s]|
Column Pair Trends Score: 99.76%

Overall Score (Average): 96.99%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 232.90it/s]|
Column Shapes Score: 93.89%

(2/2) Evaluating Column Pair Trends: |█

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 233.96it/s]|
Column Shapes Score: 88.91%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 591.20it/s]|
Column Pair Trends Score: 93.92%

Overall Score (Average): 91.41%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 235.30it/s]|
Column Shapes Score: 92.17%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 594.28it/s]|
Column Pair Trends Score: 97.88%

Overall Score (Average): 95.03%

##################################### Running CRDA #####################################


Best trial: 17. Best value: 0.000742023: 100%|██████████| 30/30 [02:45<00:00,  5.53s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 227.48it/s]|
Column Shapes Score: 96.37%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 580.28it/s]|
Column Pair Trends Score: 99.85%

Overall Score (Average): 98.11%

##################################### Done #####################################
mse_c_mixup: 0.0007993470805780787, aug_mse_c_mixup: 0.0008187312584444028, delta_percent_c_mixup: 2.4250013964279056, score_c_mixup: 0.9600265038130065
mse_ada: 0.0007993470805780787, aug_mse_ada: 0.0008151138991191707, delta_percent_ada: 1.9724621411877412, score_ada: 0.9699180122929032
mse_tabddpm: 0.0007993470805780787, aug_mse_tabddpm: 0.0010214550909402387, delta_percent_tabddpm: 27.786178965154175, score_tabddpm: 0.9599391899882384
mse_ctgan: 0.0007993470805780787, aug_mse_ctgan: 0.0019777004297970024, delta_percent_ctgan: 147.41448087440966, score_ctgan: 0.9141419624800725
mse_tvae: 0.0007993470805780787, aug_mse_tvae: 0.00

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 269.83it/s]|
Column Shapes Score: 96.62%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 630.72it/s]|
Column Pair Trends Score: 99.83%

Overall Score (Average): 98.23%

##################################### Done #####################################
mse_c_mixup: 0.000834893226389427, aug_mse_c_mixup: 0.0008074544795474748, delta_percent_c_mixup: -3.286497719069249, score_c_mixup: 0.9630939768233527
mse_ada: 0.000834893226389427, aug_mse_ada: 0.0008620829716011968, delta_percent_ada: 3.2566733508372443, score_ada: 0.9705945589641249
mse_tabddpm: 0.000834893226389427, aug_mse_tabddpm: 0.0009504791815270416, delta_percent_tabddpm: 13.84439967700741, score_tabddpm: 0.9622950530422913
mse_ctgan: 0.000834893226389427, aug_mse_ctgan: 0.0017285931127697305, delta_percent_ctgan: 107.04361445656845, score_ctgan: 0.9104180685809099
mse_tvae: 0.000834893226389427, aug_mse_tvae: 0.00104070

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 246.13it/s]|
Column Shapes Score: 88.62%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 605.37it/s]|
Column Pair Trends Score: 94.46%

Overall Score (Average): 91.54%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 198.83it/s]|
Column Shapes Score: 91.29%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 551.21it/s]|
Column Pair Trends Score: 98.04%

Overall Score (Average): 94.66%

##################################### Running CRDA #####################################


Best trial: 23. Best value: 0.000752098: 100%|██████████| 30/30 [02:31<00:00,  5.07s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 580.67it/s]|
Column Shapes Score: 99.3%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 633.14it/s]|
Column Pair Trends Score: 99.9%

Overall Score (Average): 99.6%

##################################### Done #####################################
mse_c_mixup: 0.0008158333797216775, aug_mse_c_mixup: 0.0008301595133854688, delta_percent_c_mixup: 1.7560121980641104, score_c_mixup: 0.9688361386467639
mse_ada: 0.0008158333797216775, aug_mse_ada: 0.0007922923205086675, delta_percent_ada: -2.885522926389831, score_ada: 0.9707953414755868
mse_tabddpm: 0.0008158333797216775, aug_mse_tabddpm: 0.0015160277312773343, delta_percent_tabddpm: 85.82565618907736, score_tabddpm: 0.960265919161652
mse_ctgan: 0.0008158333797216775, aug_mse_ctgan: 0.0015234765070711563, delta_percent_ctgan: 86.73868279217652, score_ctgan: 0.9153820554152164
mse_tvae: 0.0008158333797216775, aug_mse_tvae: 0.00129624

In [29]:
config = Config(
    baseline="mlp",
    dataset_path="../data/294_satellite_image.csv",
    results_dir="../experiments_all_baselines/294_satellite_image",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 289.83it/s]|
Column Shapes Score: 94.41%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 674.56it/s]|
Column Pair Trends Score: 99.87%

Overall Score (Average): 97.14%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 766.09it/s]|
Column Shapes Score: 98.14%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 781.22it/s]|
Column Pair Trends Score: 99.78%

Overall Score (Average): 98.96%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 369.47it/s]|
Column Shapes Score: 86.33%

(2/2) Evaluating Column Pair Trend

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 281.91it/s]|
Column Shapes Score: 93.32%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 741.23it/s]|
Column Pair Trends Score: 87.1%

Overall Score (Average): 90.21%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 306.52it/s]|
Column Shapes Score: 96.31%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 771.15it/s]|
Column Pair Trends Score: 98.82%

Overall Score (Average): 97.56%

##################################### Running CRDA #####################################


Best trial: 4. Best value: 0.0126855: 100%|██████████| 30/30 [03:01<00:00,  6.05s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 361.75it/s]|
Column Shapes Score: 99.1%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 750.85it/s]|
Column Pair Trends Score: 99.66%

Overall Score (Average): 99.38%

##################################### Done #####################################
mse_c_mixup: 0.016586259988476845, aug_mse_c_mixup: 0.017976189003066998, delta_percent_c_mixup: 8.380002577770963, score_c_mixup: 0.9713742828918305
mse_ada: 0.016586259988476845, aug_mse_ada: 0.01786795888370174, delta_percent_ada: 7.727473801299054, score_ada: 0.9896055380550232
mse_tabddpm: 0.016586259988476845, aug_mse_tabddpm: 0.02018977181224137, delta_percent_tabddpm: 21.725885318739927, score_tabddpm: 0.9009934152167658
mse_ctgan: 0.016586259988476845, aug_mse_ctgan: 0.026983277311236512, delta_percent_ctgan: 62.684519174201434, score_ctgan: 0.9021174628669038
mse_tvae: 0.016586259988476845, aug_mse_tvae: 0.01913865471334

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 380.75it/s]|
Column Shapes Score: 99.51%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 822.09it/s]|
Column Pair Trends Score: 99.78%

Overall Score (Average): 99.65%

##################################### Done #####################################
mse_c_mixup: 0.01831501637169458, aug_mse_c_mixup: 0.017427077624709592, delta_percent_c_mixup: -4.848146072952886, score_c_mixup: 0.9785595289072981
mse_ada: 0.01831501637169458, aug_mse_ada: 0.018703804773770826, delta_percent_ada: 2.122784900575399, score_ada: 0.9886346993414756
mse_tabddpm: 0.01831501637169458, aug_mse_tabddpm: 0.017482598054191386, delta_percent_tabddpm: -4.545004495817311, score_tabddpm: 0.9014718183525274
mse_ctgan: 0.01831501637169458, aug_mse_ctgan: 0.029356700883175924, delta_percent_ctgan: 60.28760382953302, score_ctgan: 0.8999861872497869
mse_tvae: 0.01831501637169458, aug_mse_tvae: 0.0217996539195932

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 259.84it/s]|
Column Shapes Score: 92.89%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 803.55it/s]|
Column Pair Trends Score: 86.6%

Overall Score (Average): 89.74%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 289.66it/s]|
Column Shapes Score: 96.0%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 752.90it/s]|
Column Pair Trends Score: 98.05%

Overall Score (Average): 97.02%

##################################### Running CRDA #####################################


Best trial: 5. Best value: 0.0130307: 100%|██████████| 30/30 [02:05<00:00,  4.17s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 37/37 [00:00<00:00, 387.22it/s]|
Column Shapes Score: 99.43%

(2/2) Evaluating Column Pair Trends: |██████████| 666/666 [00:00<00:00, 788.79it/s]|
Column Pair Trends Score: 99.68%

Overall Score (Average): 99.55%

##################################### Done #####################################
mse_c_mixup: 0.01435521140238501, aug_mse_c_mixup: 0.014295077556277245, delta_percent_c_mixup: -0.41889906335879573, score_c_mixup: 0.9841237961701559
mse_ada: 0.01435521140238501, aug_mse_ada: 0.015924119345702165, delta_percent_ada: 10.929187312814443, score_ada: 0.98975488881812
mse_tabddpm: 0.01435521140238501, aug_mse_tabddpm: 0.014912493213522547, delta_percent_tabddpm: 3.882087107717194, score_tabddpm: 0.9005230319935555
mse_ctgan: 0.01435521140238501, aug_mse_ctgan: 0.02302585023072366, delta_percent_ctgan: 60.4006349004243, score_ctgan: 0.8974374966005152
mse_tvae: 0.01435521140238501, aug_mse_tvae: 0.018349417471821276

In [30]:
config = Config(
    baseline="mlp",
    dataset_path="../data/503_wind.csv",
    results_dir="../experiments_all_baselines/503_wind",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 253.06it/s]|
Column Shapes Score: 95.1%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 694.77it/s]|
Column Pair Trends Score: 99.93%

Overall Score (Average): 97.52%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 500.66it/s]|
Column Shapes Score: 98.8%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 441.94it/s]|
Column Pair Trends Score: 99.74%

Overall Score (Average): 99.27%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 503.71it/s]|
Column Shapes Score: 98.02%

(2/2) Evaluating Column Pair Trends:

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 268.04it/s]|
Column Shapes Score: 89.72%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 699.67it/s]|
Column Pair Trends Score: 85.87%

Overall Score (Average): 87.79%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 350.74it/s]|
Column Shapes Score: 94.12%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 621.54it/s]|
Column Pair Trends Score: 94.77%

Overall Score (Average): 94.45%

##################################### Running CRDA #####################################


No features are uncorrelated with Z.
No candidate features found for 503_wind. Ignoring filter and proceeding with the experiment anyways.
Best trial: 6. Best value: 0.00559696: 100%|██████████| 30/30 [01:37<00:00,  3.24s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 301.43it/s]|
Column Shapes Score: 99.36%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 718.96it/s]|
Column Pair Trends Score: 99.86%

Overall Score (Average): 99.61%

##################################### Done #####################################
mse_c_mixup: 0.005887213827067298, aug_mse_c_mixup: 0.005774232217000454, delta_percent_c_mixup: -1.919101520440708, score_c_mixup: 0.97516730572786
mse_ada: 0.005887213827067298, aug_mse_ada: 0.0062391889027318325, delta_percent_ada: 5.978635836977406, score_ada: 0.9926986557163885
mse_tabddpm: 0.005887213827067298, aug_mse_tabddpm: 0.00509358201169702, delta_percent_tabddpm: -13.480601158419681, score_tabddpm: 0.9777380388615982
mse_ctgan: 0.005887213827067298, aug_mse_ctgan: 0.006027463190902282, delta_percent_ctgan: 2.3822705944562066, score_ctgan: 0.8779492921077796
mse_tvae: 0.005887213827067298, aug_mse_tvae: 0.00610149859

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 282.94it/s]|
Column Shapes Score: 98.4%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 665.25it/s]|
Column Pair Trends Score: 99.07%

Overall Score (Average): 98.74%

##################################### Done #####################################
mse_c_mixup: 0.006037843100674561, aug_mse_c_mixup: 0.00656218627060626, delta_percent_c_mixup: 8.684279488367597, score_c_mixup: 0.9801265774639736
mse_ada: 0.006037843100674561, aug_mse_ada: 0.007094239168221253, delta_percent_ada: 17.496249073260444, score_ada: 0.9913681289551783
mse_tabddpm: 0.006037843100674561, aug_mse_tabddpm: 0.006111330766860989, delta_percent_tabddpm: 1.217117850879854, score_tabddpm: 0.9799593354544662
mse_ctgan: 0.006037843100674561, aug_mse_ctgan: 0.0067991821092393415, delta_percent_ctgan: 12.60945334070907, score_ctgan: 0.8777213562476716
mse_tvae: 0.006037843100674561, aug_mse_tvae: 0.0069877805385

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 283.45it/s]|
Column Shapes Score: 92.44%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 711.00it/s]|
Column Pair Trends Score: 85.35%

Overall Score (Average): 88.9%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 361.87it/s]|
Column Shapes Score: 93.23%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 651.90it/s]|
Column Pair Trends Score: 96.45%

Overall Score (Average): 94.84%

##################################### Running CRDA #####################################


Best trial: 13. Best value: 0.00527585: 100%|██████████| 30/30 [01:31<00:00,  3.06s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 214.05it/s]|
Column Shapes Score: 99.14%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 696.55it/s]|
Column Pair Trends Score: 99.37%

Overall Score (Average): 99.26%

##################################### Done #####################################
mse_c_mixup: 0.005665833906411731, aug_mse_c_mixup: 0.006324858751326672, delta_percent_c_mixup: 11.631559551527921, score_c_mixup: 0.9883292594307289
mse_ada: 0.005665833906411731, aug_mse_ada: 0.007350204962778164, delta_percent_ada: 29.728563953495325, score_ada: 0.99316111198715
mse_tabddpm: 0.005665833906411731, aug_mse_tabddpm: 0.005455699196354381, delta_percent_tabddpm: -3.7088046266155277, score_tabddpm: 0.9790463231088677
mse_ctgan: 0.005665833906411731, aug_mse_ctgan: 0.006972834896830184, delta_percent_ctgan: 23.06811339703036, score_ctgan: 0.8889835924335496
mse_tvae: 0.005665833906411731, aug_mse_tvae: 0.00616570374

In [31]:
config = Config(
    baseline="mlp",
    dataset_path="../data/ParkinsonsTelemonitoring.csv",
    results_dir="../experiments_all_baselines/ParkinsonsTelemonitoring",
    hyperparam_tune=False,
    method_param_tune=True,
    ignore_filter=True,
    num_seeds=0,
    random_seed=0,
)
run_comparison(config)

##################################### Running C-Mixup #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 43.37it/s]|
Column Shapes Score: 94.41%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 655.92it/s]|
Column Pair Trends Score: 99.92%

Overall Score (Average): 97.17%

##################################### Running ADA #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 132.92it/s]|
Column Shapes Score: 98.17%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 660.92it/s]|
Column Pair Trends Score: 99.73%

Overall Score (Average): 98.95%

##################################### Running TabDDPM #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 68.64it/s]|
Column Shapes Score: 96.2%

(2/2) Evaluating Column Pair Trends: 

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 29.40it/s]|
Column Shapes Score: 91.25%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 692.56it/s]|
Column Pair Trends Score: 88.9%

Overall Score (Average): 90.07%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 38.64it/s]|
Column Shapes Score: 93.34%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 405.09it/s]|
Column Pair Trends Score: 96.85%

Overall Score (Average): 95.09%

##################################### Running CRDA #####################################


Best trial: 18. Best value: 0.000224936: 100%|██████████| 30/30 [02:03<00:00,  4.11s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 559.21it/s]|
Column Shapes Score: 99.83%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 690.31it/s]|
Column Pair Trends Score: 99.99%

Overall Score (Average): 99.91%

##################################### Done #####################################
mse_c_mixup: 0.0003174376178191619, aug_mse_c_mixup: 0.0006038986707163911, delta_percent_c_mixup: 90.24168429225695, score_c_mixup: 0.9716747183935901
mse_ada: 0.0003174376178191619, aug_mse_ada: 0.0003202976291184852, delta_percent_ada: 0.9009679819839828, score_ada: 0.9894776879512417
mse_tabddpm: 0.0003174376178191619, aug_mse_tabddpm: 0.0006995040623413587, delta_percent_tabddpm: 120.35953619707817, score_tabddpm: 0.9484508048814067
mse_ctgan: 0.0003174376178191619, aug_mse_ctgan: 0.005071395066941346, delta_percent_ctgan: 1497.603680931862, score_ctgan: 0.9007267656402931
mse_tvae: 0.0003174376178191619, aug_mse_tvae: 0.001

...

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 353.61it/s]|
Column Shapes Score: 99.69%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 689.75it/s]|
Column Pair Trends Score: 99.92%

Overall Score (Average): 99.81%

##################################### Done #####################################
mse_c_mixup: 0.00019545311123085488, aug_mse_c_mixup: 0.000608646098627207, delta_percent_c_mixup: 211.40261456790975, score_c_mixup: 0.9785537311155323
mse_ada: 0.00019545311123085488, aug_mse_ada: 0.0004505772077060968, delta_percent_ada: 130.52956531037668, score_ada: 0.9908069023327051
mse_tabddpm: 0.00019545311123085488, aug_mse_tabddpm: 0.0009496925001410806, delta_percent_tabddpm: 385.8927515455988, score_tabddpm: 0.9453384628900237
mse_ctgan: 0.00019545311123085488, aug_mse_ctgan: 0.0041626640351079035, delta_percent_ctgan: 2029.750715603227, score_ctgan: 0.9003538462173228
mse_tvae: 0.00019545311123085488, aug_mse_tvae: 

/Users/hosseinmohebbi/Desktop/code/CRDA/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 21.36it/s]|
Column Shapes Score: 87.65%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 684.89it/s]|
Column Pair Trends Score: 89.83%

Overall Score (Average): 88.74%

##################################### Running TVAE #####################################
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 26.67it/s]|
Column Shapes Score: 90.4%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 424.91it/s]|
Column Pair Trends Score: 97.19%

Overall Score (Average): 93.8%

##################################### Running CRDA #####################################


Best trial: 7. Best value: 0.000219253: 100%|██████████| 30/30 [02:08<00:00,  4.27s/it]


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 504.18it/s]|
Column Shapes Score: 99.84%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:00<00:00, 669.31it/s]|
Column Pair Trends Score: 99.99%

Overall Score (Average): 99.92%

##################################### Done #####################################
mse_c_mixup: 0.000460008136857942, aug_mse_c_mixup: 0.0004238345775914932, delta_percent_c_mixup: -7.86367813263699, score_c_mixup: 0.9838586465834186
mse_ada: 0.000460008136857942, aug_mse_ada: 0.00038422383262322904, delta_percent_ada: -16.474557331170946, score_ada: 0.9882748734745106
mse_tabddpm: 0.000460008136857942, aug_mse_tabddpm: 0.0007439559436944555, delta_percent_tabddpm: 61.72669222244676, score_tabddpm: 0.9486928435897863
mse_ctgan: 0.000460008136857942, aug_mse_ctgan: 0.0039819043846992044, delta_percent_ctgan: 765.6160762497298, score_ctgan: 0.8873753056234048
mse_tvae: 0.000460008136857942, aug_mse_tvae: 0.002348

In [32]:
comparison_df.to_csv("../experiments_all_baselines/comparison.csv", index=False)